# Cross-Validation Against External Labeled Datasets

**Purpose**: Validate our AMTTP production model against independently-labeled fraud datasets.  
**Models**: TX-Level Ensemble (XGBoost + LightGBM + sklearn Meta-Learner)  
**Environment**: CPU only — no GPU needed (tree inference + sklearn metrics)  

### Datasets Tested
| # | Dataset | Source | Labels | Size |
|---|---------|--------|--------|------|
| 1 | Elliptic Bitcoin | Elliptic (via Kaggle) | Law enforcement + exchange reports | 203K tx |
| 2 | XBlock Phishing | Zhejiang University | Confirmed phishing addresses | 2,330 addresses |
| 3 | Forta Network | Forta Foundation | Security bot detections (live) | Variable |
| 4 | OFAC SDN | U.S. Treasury | Sanctioned crypto addresses | ~300 addresses |
| 5 | Chainabuse | Community | Crowd-reported scam addresses | Variable |

In [1]:
# ============================================================================
# CELL 1: Setup & Install Dependencies
# ============================================================================
# Works on both Colab and local Python 3.10+

import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('☁️  Running in Google Colab')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'xgboost', 'lightgbm', 'scikit-learn', 'pandas',
                           'requests', 'kagglehub', 'web3'])
    # Upload your models to Colab or mount Google Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    MODELS_DIR = '/content/tx_level_models'  # Adjust if using Drive
    DATA_DIR = '/content/external_datasets'
else:
    print('💻 Running locally')
    MODELS_DIR = r'c:\amttp\ml\Automation\tx_level_models'
    DATA_DIR = r'c:\amttp\data\external_validation'

os.makedirs(DATA_DIR, exist_ok=True)

import numpy as np
import pandas as pd
import json
import time
import warnings
import requests
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

print(f'Models dir: {MODELS_DIR}')
print(f'Data dir:   {DATA_DIR}')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')

💻 Running locally
Models dir: c:\amttp\ml\Automation\tx_level_models
Data dir:   c:\amttp\data\external_validation
NumPy: 2.4.2, Pandas: 2.2.3


In [2]:
# ============================================================================
# CELL 2: Load Production Models
# ============================================================================
import xgboost as xgb
import lightgbm as lgb
import joblib
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, classification_report,
    confusion_matrix, precision_recall_curve
)

models_path = Path(MODELS_DIR)

# Load XGBoost
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(str(models_path / 'xgboost_tx.ubj'))
print(f'✅ XGBoost loaded ({xgb_model.n_estimators} trees)')

# Load LightGBM
lgb_model = lgb.Booster(model_file=str(models_path / 'lightgbm_tx.txt'))
print(f'✅ LightGBM loaded ({lgb_model.num_trees()} trees)')

# Load Meta-Learner
meta_model = joblib.load(str(models_path / 'meta_learner.joblib'))
print(f'✅ Meta-Learner loaded (coef={meta_model.coef_[0].tolist()})')

# Load metadata
with open(models_path / 'metadata.json') as f:
    metadata = json.load(f)
OPTIMAL_THRESHOLD = metadata['optimal_threshold']
print(f'✅ Threshold: {OPTIMAL_THRESHOLD:.4f}')

# Feature schema for our production model
FEATURE_NAMES = metadata['features']['all']  # 21 features
print(f'\n📋 Feature schema ({len(FEATURE_NAMES)} features):')
for i, f in enumerate(FEATURE_NAMES):
    print(f'  [{i:2d}] {f}')

✅ XGBoost loaded (None trees)
✅ LightGBM loaded (500 trees)
✅ Meta-Learner loaded (coef=[7.742284456450318, 2.2219675541554564])
✅ Threshold: 0.5950

📋 Feature schema (21 features):
  [ 0] value_eth
  [ 1] gas_price_gwei
  [ 2] gas_used
  [ 3] gas_limit
  [ 4] nonce
  [ 5] transaction_type
  [ 6] gas_efficiency
  [ 7] value_log
  [ 8] gas_price_log
  [ 9] is_round_amount
  [10] value_gas_ratio
  [11] is_zero_value
  [12] is_high_nonce
  [13] sender_total_transactions
  [14] sender_total_sent
  [15] sender_total_received
  [16] sender_balance
  [17] sender_avg_sent
  [18] sender_unique_receivers
  [19] sender_in_out_ratio
  [20] sender_active_duration_mins


In [3]:
# ============================================================================
# CELL 3: Prediction & Evaluation Utilities
# ============================================================================

def predict_ensemble(X: np.ndarray) -> np.ndarray:
    """
    Run our production TX-Level ensemble.
    X: [n_samples, 21] feature matrix in FEATURE_NAMES order.
    Returns: fraud probabilities [n_samples]
    """
    X = np.nan_to_num(X.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    xgb_prob = xgb_model.predict_proba(X)[:, 1]
    lgb_prob = lgb_model.predict(X)
    meta_input = np.column_stack([xgb_prob, lgb_prob])
    fraud_prob = meta_model.predict_proba(meta_input)[:, 1]
    return fraud_prob


def derive_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the 7 derived features from raw tx fields.
    Input df should have: value_eth, gas_price_gwei, gas_used, gas_limit, nonce, transaction_type
    Missing columns are filled with 0.
    """
    d = df.copy()
    for col in ['value_eth', 'gas_price_gwei', 'gas_used', 'gas_limit', 'nonce', 'transaction_type']:
        if col not in d.columns:
            d[col] = 0.0
    
    d['gas_efficiency'] = d['gas_used'] / d['gas_limit'].replace(0, 1)
    d['value_log'] = np.log1p(d['value_eth'].clip(lower=0))
    d['gas_price_log'] = np.log1p(d['gas_price_gwei'].clip(lower=0))
    d['is_round_amount'] = ((d['value_eth'] * 100) % 1 < 0.01).astype(np.float32)
    d['value_gas_ratio'] = d['value_eth'] / d['gas_price_gwei'].replace(0, 1)
    d['is_zero_value'] = (d['value_eth'] == 0).astype(np.float32)
    d['is_high_nonce'] = (d['nonce'] > 1000).astype(np.float32)
    return d


def build_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    """
    Build the 21-feature matrix from a DataFrame.
    Missing features are filled with 0 (unknown sender features).
    """
    df = derive_features(df)
    for col in FEATURE_NAMES:
        if col not in df.columns:
            df[col] = 0.0
    X = df[FEATURE_NAMES].fillna(0).values.astype(np.float32)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


def evaluate_and_report(name: str, y_true: np.ndarray, y_prob: np.ndarray,
                        threshold: float = None) -> dict:
    """
    Full evaluation report for one external dataset.
    """
    if threshold is None:
        threshold = OPTIMAL_THRESHOLD

    y_pred = (y_prob >= threshold).astype(int)

    # Handle edge cases (all same class)
    n_pos = y_true.sum()
    n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0:
        print(f'\n⚠️  {name}: Only one class present (pos={n_pos}, neg={n_neg})')
        print(f'   Mean predicted prob: {y_prob.mean():.4f}')
        print(f'   Predicted positive rate: {y_pred.mean():.2%}')
        return {'name': name, 'n_samples': len(y_true), 'n_pos': int(n_pos),
                'n_neg': int(n_neg), 'error': 'single_class'}

    roc = roc_auc_score(y_true, y_prob)
    pr = average_precision_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)

    # Also find the dataset-optimal threshold
    prec_curve, rec_curve, thresholds = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec_curve * rec_curve / (prec_curve + rec_curve + 1e-10)
    best_idx = np.argmax(f1s)
    best_thresh = thresholds[min(best_idx, len(thresholds)-1)]
    best_f1 = f1s[best_idx]

    print(f'\n{"="*70}')
    print(f'  📊 {name}')
    print(f'{"="*70}')
    print(f'  Samples: {len(y_true):,}  (pos={n_pos:,}, neg={n_neg:,}, rate={n_pos/len(y_true):.2%})')
    print(f'  ── Using production threshold ({threshold:.4f}) ──')
    print(f'    ROC-AUC:   {roc:.4f}')
    print(f'    PR-AUC:    {pr:.4f}')
    print(f'    F1:        {f1:.4f}')
    print(f'    Precision: {prec:.4f}')
    print(f'    Recall:    {rec:.4f}')
    print(f'  ── Dataset-optimal threshold ({best_thresh:.4f}) ──')
    print(f'    Best F1:   {best_f1:.4f}')
    print(f'\n  Confusion Matrix (production threshold):')
    cm = confusion_matrix(y_true, y_pred)
    print(f'    TN={cm[0,0]:,}  FP={cm[0,1]:,}')
    print(f'    FN={cm[1,0]:,}  TP={cm[1,1]:,}')

    return {
        'name': name, 'n_samples': len(y_true), 'n_pos': int(n_pos),
        'roc_auc': round(roc, 4), 'pr_auc': round(pr, 4),
        'f1_production': round(f1, 4), 'precision': round(prec, 4),
        'recall': round(rec, 4), 'f1_optimal': round(float(best_f1), 4),
        'optimal_threshold': round(float(best_thresh), 4),
    }


# Collect all results
ALL_RESULTS = []
print('✅ Utilities ready')

✅ Utilities ready


---
## Dataset 1: Elliptic Bitcoin Transaction Dataset

**Source**: Elliptic (via Kaggle)  
**Labels**: Law enforcement investigations + exchange compliance teams  
**Size**: 203,769 transactions (4,545 illicit, 42,019 licit, rest unknown)  
**Why**: Gold-standard independent labels. Tests cross-chain generalisation.

In [4]:
# ============================================================================
# CELL 4: Download Elliptic Dataset
# ============================================================================
# Option A: kagglehub (recommended)
# Option B: Manual download from https://www.kaggle.com/datasets/ellipticco/elliptic-data-set
#           and place files in DATA_DIR/elliptic/

elliptic_dir = Path(DATA_DIR) / 'elliptic'
elliptic_dir.mkdir(parents=True, exist_ok=True)

# Check if already downloaded
features_file = elliptic_dir / 'elliptic_txs_features.csv'
classes_file = elliptic_dir / 'elliptic_txs_classes.csv'
edges_file = elliptic_dir / 'elliptic_txs_edgelist.csv'

if not features_file.exists():
    print('📥 Downloading Elliptic dataset via kagglehub...')
    try:
        import kagglehub
        path = kagglehub.dataset_download('ellipticco/elliptic-data-set')
        print(f'   Downloaded to: {path}')
        
        # Copy files to our data dir
        import shutil
        dl_path = Path(path)
        for f in dl_path.rglob('*.csv'):
            dest = elliptic_dir / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                print(f'   Copied: {f.name}')
    except Exception as e:
        print(f'❌ kagglehub download failed: {e}')
        print(f'\n📋 Manual download instructions:')
        print(f'   1. Go to: https://www.kaggle.com/datasets/ellipticco/elliptic-data-set')
        print(f'   2. Download and extract to: {elliptic_dir}')
        print(f'   3. Expected files: elliptic_txs_features.csv, elliptic_txs_classes.csv, elliptic_txs_edgelist.csv')
else:
    print(f'✅ Elliptic dataset already present in {elliptic_dir}')

# Verify files
for f in [features_file, classes_file]:
    if f.exists():
        print(f'   ✓ {f.name} ({f.stat().st_size / 1e6:.1f} MB)')
    else:
        print(f'   ✗ {f.name} — MISSING')

📥 Downloading Elliptic dataset via kagglehub...


100%|██████████| 146M/146M [00:04<00:00, 32.1MB/s] 

Extracting files...


   Downloaded to: C:\Users\Administrator\.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1
   Copied: elliptic_txs_classes.csv
   Copied: elliptic_txs_edgelist.csv
   Copied: elliptic_txs_features.csv
   ✓ elliptic_txs_features.csv (689.7 MB)
   ✓ elliptic_txs_classes.csv (3.3 MB)


In [5]:
# ============================================================================
# CELL 5: Evaluate on Elliptic Dataset
# ============================================================================
print('📊 Evaluating on Elliptic Bitcoin Dataset...')
t0 = time.time()

if features_file.exists() and classes_file.exists():
    # Load features (no header, 167 columns: txId + 166 features)
    # Features 1-93: local (tx-level), 94-166: aggregated (1-hop neighborhood)
    df_feat = pd.read_csv(features_file, header=None)
    df_feat.columns = ['txId'] + [f'feat_{i}' for i in range(1, 167)]
    
    # Load classes
    df_class = pd.read_csv(classes_file)
    df_class.columns = ['txId', 'class']
    
    # Merge
    df_ell = df_feat.merge(df_class, on='txId')
    
    # Filter to labeled only (1=illicit, 2=licit, unknown=unlabeled)
    df_labeled = df_ell[df_ell['class'].isin(['1', '2', 1, 2])].copy()
    df_labeled['fraud'] = (df_labeled['class'].astype(str) == '1').astype(int)
    
    print(f'  Total transactions: {len(df_ell):,}')
    print(f'  Labeled transactions: {len(df_labeled):,}')
    print(f'  Illicit: {df_labeled["fraud"].sum():,}  |  Licit: {(df_labeled["fraud"]==0).sum():,}')
    print(f'  Fraud rate: {df_labeled["fraud"].mean():.2%}')
    
    # ── Feature Mapping ──
    # Elliptic has anonymised features, so we can't map exactly.
    # Strategy: Use their features directly as a proxy.
    # The Elliptic paper states:
    #   feat_1 = time step
    #   feat_2-94 = local features (tx amount, fee, in/out degree, etc.)
    #   feat_95-166 = 1-hop aggregated features
    # 
    # Since features are anonymised, we create a BEST-EFFORT mapping:
    #   - feat_2 → likely relates to "value" (tx amount)
    #   - feat_3-5 → likely gas/fee related
    #   - Features involving degree/neighbor counts → sender aggregate proxies
    #
    # TWO evaluation strategies:
    #   A) Map Elliptic features to our 21-feature schema (feature-level transfer)
    #   B) Train a thin adapter (logistic regression) on Elliptic features using
    #      our model's predictions as a score — tests if our model's "fraud concept"
    #      aligns with Elliptic's ground truth
    
    # === Strategy A: Best-effort feature mapping ===
    # Map anonymised Elliptic features to our schema
    # Based on the Elliptic paper (Weber et al., 2019):
    #   Local features include: # inputs, # outputs, tx fee, output volume,
    #   aggregate of input/output values, avg input/output value, etc.
    
    mapped = pd.DataFrame()
    # Using first few local features as proxies for tx characteristics
    mapped['value_eth'] = df_labeled['feat_2'].abs()       # Likely amount-related
    mapped['gas_price_gwei'] = df_labeled['feat_3'].abs()  # Likely fee-related  
    mapped['gas_used'] = df_labeled['feat_4'].abs()        # Proxy
    mapped['gas_limit'] = df_labeled['feat_5'].abs()       # Proxy
    mapped['nonce'] = df_labeled['feat_6'].abs()           # Proxy for sequence
    mapped['transaction_type'] = 0.0                        # Not available in Bitcoin
    # Sender aggregates from Elliptic's 1-hop features
    mapped['sender_total_transactions'] = df_labeled['feat_95'].abs()
    mapped['sender_total_sent'] = df_labeled['feat_96'].abs()
    mapped['sender_total_received'] = df_labeled['feat_97'].abs()
    mapped['sender_balance'] = df_labeled['feat_98'].abs()
    mapped['sender_avg_sent'] = df_labeled['feat_99'].abs()
    mapped['sender_unique_receivers'] = df_labeled['feat_100'].abs()
    mapped['sender_in_out_ratio'] = df_labeled['feat_101'].abs()
    mapped['sender_active_duration_mins'] = df_labeled['feat_102'].abs()
    
    X_ell = build_feature_matrix(mapped)
    y_ell = df_labeled['fraud'].values.astype(int)
    
    # Predict
    prob_ell = predict_ensemble(X_ell)
    
    result_ell_mapped = evaluate_and_report(
        'Elliptic (feature-mapped)', y_ell, prob_ell
    )
    ALL_RESULTS.append(result_ell_mapped)
    
    # === Strategy B: Use all 166 Elliptic features directly ===
    # Train a small probe to test if our model's score correlates with true labels
    # This tests the "concept alignment" — does our model's notion of fraud
    # match law enforcement labels?
    from sklearn.model_selection import cross_val_predict
    from sklearn.linear_model import LogisticRegression
    
    # Create hybrid features: Elliptic original + our model's score
    feat_cols = [c for c in df_labeled.columns if c.startswith('feat_')]
    X_ell_native = df_labeled[feat_cols].values.astype(np.float32)
    X_ell_native = np.nan_to_num(X_ell_native, nan=0.0)
    
    # Our model's predictions as an additional feature
    our_scores = prob_ell.reshape(-1, 1)
    
    # Test: Does adding our score improve a simple classifier on Elliptic?
    print('\n  ── Concept Alignment Test ──')
    print('  Testing if our fraud score adds value to Elliptic native features...')
    
    # Baseline: Elliptic features only
    lr_base = LogisticRegression(max_iter=1000, C=0.1, random_state=42)
    base_preds = cross_val_predict(lr_base, X_ell_native, y_ell, cv=5, method='predict_proba')[:, 1]
    base_roc = roc_auc_score(y_ell, base_preds)
    base_ap = average_precision_score(y_ell, base_preds)
    
    # With our score
    X_augmented = np.column_stack([X_ell_native, our_scores])
    lr_aug = LogisticRegression(max_iter=1000, C=0.1, random_state=42)
    aug_preds = cross_val_predict(lr_aug, X_augmented, y_ell, cv=5, method='predict_proba')[:, 1]
    aug_roc = roc_auc_score(y_ell, aug_preds)
    aug_ap = average_precision_score(y_ell, aug_preds)
    
    print(f'  Elliptic features only:     ROC-AUC={base_roc:.4f}  PR-AUC={base_ap:.4f}')
    print(f'  Elliptic + our score:       ROC-AUC={aug_roc:.4f}  PR-AUC={aug_ap:.4f}')
    delta = aug_roc - base_roc
    if delta > 0.005:
        print(f'  ✅ Our score adds +{delta:.4f} ROC-AUC — complementary fraud signal detected')
    elif delta > -0.005:
        print(f'  ➡️  Our score is neutral ({delta:+.4f}) — possibly redundant with Elliptic features')
    else:
        print(f'  ⚠️  Our score reduces performance ({delta:+.4f}) — possible domain mismatch')
    
    # Direct correlation: Do our high-risk predictions overlap with illicit labels?
    our_high_risk = prob_ell >= OPTIMAL_THRESHOLD
    overlap_rate = y_ell[our_high_risk].mean() if our_high_risk.sum() > 0 else 0
    print(f'\n  ── Overlap Analysis ──')
    print(f'  Our model flags {our_high_risk.sum():,} / {len(y_ell):,} as high-risk ({our_high_risk.mean():.2%})')
    print(f'  Of those, {overlap_rate:.2%} are truly illicit per Elliptic labels')
    print(f'  Baseline fraud rate: {y_ell.mean():.2%}')
    lift = overlap_rate / max(y_ell.mean(), 1e-6)
    print(f'  Lift: {lift:.1f}x over baseline')
    
    print(f'\n  ⏱️  Completed in {time.time()-t0:.1f}s')
else:
    print('⚠️  Elliptic dataset not found. Download it first (Cell 4).')

📊 Evaluating on Elliptic Bitcoin Dataset...
  Total transactions: 203,769
  Labeled transactions: 46,564
  Illicit: 4,545  |  Licit: 42,019
  Fraud rate: 9.76%

  📊 Elliptic (feature-mapped)
  Samples: 46,564  (pos=4,545, neg=42,019, rate=9.76%)
  ── Using production threshold (0.5950) ──
    ROC-AUC:   0.3484
    PR-AUC:    0.0688
    F1:        0.0025
    Precision: 0.0042
    Recall:    0.0018
  ── Dataset-optimal threshold (0.0024) ──
    Best F1:   0.1795

  Confusion Matrix (production threshold):
    TN=40,109  FP=1,910
    FN=4,537  TP=8

  ── Concept Alignment Test ──
  Testing if our fraud score adds value to Elliptic native features...
  Elliptic features only:     ROC-AUC=0.9311  PR-AUC=0.6241
  Elliptic + our score:       ROC-AUC=0.9334  PR-AUC=0.6481
  ➡️  Our score is neutral (+0.0024) — possibly redundant with Elliptic features

  ── Overlap Analysis ──
  Our model flags 1,918 / 46,564 as high-risk (4.12%)
  Of those, 0.42% are truly illicit per Elliptic labels
  Baseli

---
## Dataset 2: XBlock Ethereum Phishing Addresses

**Source**: Zhejiang University (xblock.pro)  
**Labels**: Confirmed phishing addresses vs normal  
**Size**: ~2,330 addresses (1,165 phishing + 1,165 normal)  
**Why**: Ethereum-native, independently labeled phishing accounts

In [4]:
# ============================================================================
# CELL 6: Download & Evaluate XBlock Phishing Dataset
# ============================================================================
print('📊 Evaluating on XBlock Ethereum Phishing Dataset...')
t0 = time.time()

xblock_dir = Path(DATA_DIR) / 'xblock'
xblock_dir.mkdir(parents=True, exist_ok=True)

xblock_file = xblock_dir / 'transaction_dataset.csv'

if not xblock_file.exists():
    print('📥 Attempting to download via kagglehub...')
    try:
        import kagglehub
        kpath = kagglehub.dataset_download('vagifa/ethereum-frauddetection-dataset')
        print(f'   Downloaded to: {kpath}')
        # Find the CSV
        import shutil
        for f in Path(kpath).rglob('*.csv'):
            dest = xblock_dir / f.name
            shutil.copy2(f, dest)
            print(f'   Copied: {f.name}')
            if 'transaction' in f.name.lower():
                xblock_file = dest
    except Exception as e:
        print(f'   ⚠️  kagglehub failed: {e}')
        print(f'   Manual: download from https://www.kaggle.com/datasets/vagifa/ethereum-frauddetection-dataset')
        print(f'   Save as: {xblock_file}')

if xblock_file.exists():
    df_xb = pd.read_csv(xblock_file)
    print(f'  Loaded {len(df_xb):,} records')
    print(f'  Columns: {list(df_xb.columns[:15])}...' if len(df_xb.columns) > 15 else f'  Columns: {list(df_xb.columns)}')
    
    # Identify label column
    label_col = None
    for candidate in ['FLAG', 'flag', 'label', 'Label', 'class', 'Class', 'is_phishing', 'FLAG ']:
        if candidate in df_xb.columns:
            label_col = candidate
            break
    
    if label_col is None:
        print(f'  ⚠️  Could not identify label column among: {list(df_xb.columns)}')
        print(df_xb.head(3))
    else:
        y_xb = df_xb[label_col].astype(int).values
        print(f'  Label column: "{label_col}"')
        print(f'  Phishing: {y_xb.sum():,}  |  Normal: {(y_xb==0).sum():,}')
        
        # Map XBlock features to our schema
        feature_map = {
            'Avg min between sent tnx': 'sender_active_duration_mins',
            'Sent tnx': 'sender_total_transactions',
            'max value received ': 'sender_total_received',
            'avg val sent': 'sender_avg_sent',
            'total Ether sent': 'sender_total_sent',
            'total ether received': 'sender_total_received',
            'total ether balance': 'sender_balance',
            'ERC20 uniq sent addr': 'sender_unique_receivers',
            'Unique Sent To Addresses': 'sender_unique_receivers',
        }
        
        mapped_xb = pd.DataFrame()
        for src_col, dst_col in feature_map.items():
            if dst_col and src_col in df_xb.columns:
                mapped_xb[dst_col] = pd.to_numeric(df_xb[src_col], errors='coerce').fillna(0)
        
        if 'sender_total_sent' in mapped_xb.columns and 'sender_total_transactions' in mapped_xb.columns:
            mapped_xb['value_eth'] = mapped_xb['sender_total_sent'] / mapped_xb['sender_total_transactions'].replace(0, 1)
        else:
            mapped_xb['value_eth'] = 0.0
        
        if 'Sent tnx' in df_xb.columns and 'Received Tnx' in df_xb.columns:
            sent = pd.to_numeric(df_xb['Sent tnx'], errors='coerce').fillna(0)
            recv = pd.to_numeric(df_xb['Received Tnx'], errors='coerce').fillna(0)
            mapped_xb['sender_in_out_ratio'] = recv / sent.replace(0, 1)
        
        X_xb = build_feature_matrix(mapped_xb)
        
        mapped_count = sum(1 for c in FEATURE_NAMES if c in mapped_xb.columns and (mapped_xb[c] != 0).any())
        print(f'  Features mapped: {mapped_count}/{len(FEATURE_NAMES)}')
        
        prob_xb = predict_ensemble(X_xb)
        result_xb = evaluate_and_report('XBlock Phishing', y_xb, prob_xb)
        ALL_RESULTS.append(result_xb)
        
        print(f'\n  ⏱️  Completed in {time.time()-t0:.1f}s')
else:
    print('⚠️  XBlock dataset not found. See download instructions above.')

📊 Evaluating on XBlock Ethereum Phishing Dataset...
  Loaded 9,841 records
  Columns: ['Unnamed: 0', 'Index', 'Address', 'FLAG', 'Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 'min value received', 'max value received ', 'avg val received']...
  Label column: "FLAG"
  Phishing: 2,179  |  Normal: 7,662
  Features mapped: 9/21

  📊 XBlock Phishing
  Samples: 9,841  (pos=2,179, neg=7,662, rate=22.14%)
  ── Using production threshold (0.5950) ──
    ROC-AUC:   0.6024
    PR-AUC:    0.3710
    F1:        0.0000
    Precision: 0.0000
    Recall:    0.0000
  ── Dataset-optimal threshold (0.0404) ──
    Best F1:   0.3956

  Confusion Matrix (production threshold):
    TN=7,619  FP=43
    FN=2,179  TP=0

  ⏱️  Completed in 0.5s


In [5]:
# ============================================================================
# CELL 6b: Compare TX-Level vs Student v2 Address-Level on XBlock
# ============================================================================
# The TX-level model uses 21 features (8 sender + 6 tx-raw + 7 derived)
# The Student v2 model uses 71 features (5 sender + 64 VAE latents + recon_err + graphsage_score)
# XBlock only has address aggregates — let's test BOTH models fairly.

import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from scipy.special import expit
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, confusion_matrix
import joblib

print('='*80)
print('  COMPARISON: TX-Level vs Student v2 Address-Level on XBlock Phishing')
print('='*80)

# ── Load Student v2 address-level models ──
STUDENT_DIR = Path(r'c:\amttp\ml\Automation\amttp_models_20251231_174617')

# Load Student v2 feature config
student_feat_config = json.load(open(STUDENT_DIR / 'feature_config.json'))
student_features = student_feat_config.get('boost_features', student_feat_config.get('features', []))
print(f'\nStudent v2 feature config: {len(student_features)} features')
print(f'  First 10: {student_features[:10]}')

# Load Student v2 XGBoost
student_xgb = xgb.XGBClassifier()
student_xgb.load_model(str(STUDENT_DIR / 'xgboost_fraud.ubj'))
print(f'✅ Student v2 XGBoost loaded')

# Load Student v2 LightGBM
student_lgb = lgb.Booster(model_file=str(STUDENT_DIR / 'lightgbm_fraud.txt'))
print(f'✅ Student v2 LightGBM loaded ({student_lgb.num_trees()} trees)')

# Student v2 meta-learner coefficients (from integration_service.py)
STUDENT_META_COEF = np.array([0.7021, 3.3994, 0.4668, -2.5898, 0.8492, -1.6858, 8.9820])
STUDENT_META_INTERCEPT = np.array([-5.5341])

# ── Map XBlock features to Student v2 schema ──
# Student v2 expects 71 features: 5 tabular + 64 VAE latents + recon_err + graphsage_score
# From feature_config: sender_total_transactions, sender_total_sent, sender_total_received,
#   sender_sophisticated_score, sender_hybrid_score, vae_z0..z63, recon_err, graphsage_score

student_mapped = pd.DataFrame()

# Map available XBlock columns to Student v2's 5 tabular features
student_mapping = {
    'Sent tnx': 'sender_total_transactions', 
    'total Ether sent': 'sender_total_sent',
    'total ether received': 'sender_total_received',
}

for src, dst in student_mapping.items():
    if src in df_xb.columns:
        student_mapped[dst] = pd.to_numeric(df_xb[src], errors='coerce').fillna(0)

# sender_sophisticated_score and sender_hybrid_score are composite scores 
# from the teacher pipeline — XBlock doesn't have them, so default to 0
student_mapped['sender_sophisticated_score'] = 0.0
student_mapped['sender_hybrid_score'] = 0.0

# Build the full 71-feature matrix (zero-pad VAE/GNN slots)
X_student = np.zeros((len(df_xb), len(student_features)))
for i, feat_name in enumerate(student_features):
    if feat_name in student_mapped.columns:
        X_student[:, i] = student_mapped[feat_name].values

print(f'\nStudent v2 feature matrix: {X_student.shape}')
populated = sum(1 for i, f in enumerate(student_features) if (X_student[:, i] != 0).any())
print(f'  Populated features: {populated}/{len(student_features)}')

# ── Predict with Student v2 ──
# XGBoost predict
try:
    student_xgb_prob = student_xgb.predict_proba(X_student)[:, 1]
except Exception:
    dmat = xgb.DMatrix(X_student, feature_names=[f'f{i}' for i in range(X_student.shape[1])])
    student_xgb_prob = student_xgb.predict(dmat)
    if hasattr(student_xgb_prob, 'shape') and len(student_xgb_prob.shape) == 2:
        student_xgb_prob = student_xgb_prob[:, 1]

# LightGBM predict
student_lgb_prob = student_lgb.predict(X_student)

print(f'  Student XGB probs: mean={student_xgb_prob.mean():.4f}, std={student_xgb_prob.std():.4f}')
print(f'  Student LGB probs: mean={student_lgb_prob.mean():.4f}, std={student_lgb_prob.std():.4f}')

# Student v2 meta-ensemble (7 input features — from integration_service.py)
# The 7 features are: xgb_prob, lgb_prob, gatv2_score, graphsage_score, recon_err, vae_score, heuristic
# We only have xgb + lgb, rest are zero
student_meta_input = np.column_stack([
    student_xgb_prob,
    student_lgb_prob,
    np.zeros(len(df_xb)),  # gatv2_score
    np.zeros(len(df_xb)),  # graphsage_score  
    np.zeros(len(df_xb)),  # recon_err
    np.zeros(len(df_xb)),  # vae_score
    np.zeros(len(df_xb)),  # heuristic
])
student_logit = student_meta_input @ STUDENT_META_COEF + STUDENT_META_INTERCEPT
student_prob = expit(student_logit)

print(f'  Student ensemble probs: mean={student_prob.mean():.4f}, std={student_prob.std():.4f}')

# ── Evaluate both models ──
y_true = y_xb  # XBlock labels

def eval_model(name, y_true, probs, threshold=0.5):
    roc = roc_auc_score(y_true, probs)
    pr = average_precision_score(y_true, probs)
    preds = (probs >= threshold).astype(int)
    f1 = f1_score(y_true, preds, zero_division=0)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    
    # Find optimal threshold
    best_f1, best_t = 0, 0
    for t in np.arange(0.01, 0.99, 0.01):
        f = f1_score(y_true, (probs >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    
    return {'roc': roc, 'pr': pr, 'f1': f1, 'prec': prec, 'rec': rec, 'best_f1': best_f1, 'best_t': best_t}

# TX-Level model (already ran, use existing prob_xb)
tx_result = eval_model('TX-Level', y_true, prob_xb, threshold=OPTIMAL_THRESHOLD)

# Student v2 address-level
student_result = eval_model('Student v2', y_true, student_prob, threshold=0.5)

# Student v2 individual models 
student_xgb_result = eval_model('Student XGB only', y_true, student_xgb_prob, threshold=0.5)
student_lgb_result = eval_model('Student LGB only', y_true, student_lgb_prob, threshold=0.5)

# ── Print comparison table ──
print(f'\n{"="*90}')
print(f'  XBlock Phishing Dataset — Model Comparison (N={len(y_true):,}, fraud rate={y_true.mean():.1%})')
print(f'{"="*90}')
print(f'{"Model":<30} {"ROC-AUC":>8} {"PR-AUC":>8} {"F1":>8} {"Best F1":>8} {"Best θ":>8} {"Feats Used":>10}')
print(f'{"─"*90}')
print(f'{"TX-Level Ensemble (prod)":<30} {tx_result["roc"]:>8.4f} {tx_result["pr"]:>8.4f} {tx_result["f1"]:>8.4f} {tx_result["best_f1"]:>8.4f} {tx_result["best_t"]:>8.4f} {"9/21":>10}')
print(f'{"Student v2 XGB (addr)":<30} {student_xgb_result["roc"]:>8.4f} {student_xgb_result["pr"]:>8.4f} {student_xgb_result["f1"]:>8.4f} {student_xgb_result["best_f1"]:>8.4f} {student_xgb_result["best_t"]:>8.4f} {"3/71":>10}')
print(f'{"Student v2 LGB (addr)":<30} {student_lgb_result["roc"]:>8.4f} {student_lgb_result["pr"]:>8.4f} {student_lgb_result["f1"]:>8.4f} {student_lgb_result["best_f1"]:>8.4f} {student_lgb_result["best_t"]:>8.4f} {"3/71":>10}')
print(f'{"Student v2 Ensemble (addr)":<30} {student_result["roc"]:>8.4f} {student_result["pr"]:>8.4f} {student_result["f1"]:>8.4f} {student_result["best_f1"]:>8.4f} {student_result["best_t"]:>8.4f} {"3/71":>10}')
print(f'{"Random baseline":<30} {"0.5000":>8} {y_true.mean():>8.4f} {"—":>8} {"—":>8} {"—":>8} {"—":>10}')
print(f'{"─"*90}')

print(f'\n📋 KEY INSIGHT:')
print(f'   TX-Level model with sender features only: ROC-AUC = {tx_result["roc"]:.4f}')
print(f'   Student v2 address model:                 ROC-AUC = {student_result["roc"]:.4f}')
if tx_result['roc'] > student_result['roc']:
    print(f'   ➡️  TX-Level production model is BETTER even with fewer usable features')
    print(f'      (8 raw sender aggregates > 3 of 71 padded features + missing VAE/GNN)')
else:
    print(f'   ➡️  Student v2 address model is BETTER on address-level data')
    print(f'      This makes sense — it was trained on address-level features')
print(f'   Both above random (0.50) = fraud signal transfers to independent dataset ✓')

  COMPARISON: TX-Level vs Student v2 Address-Level on XBlock Phishing

Student v2 feature config: 71 features
  First 10: ['sender_total_transactions', 'sender_total_sent', 'sender_total_received', 'sender_sophisticated_score', 'sender_hybrid_score', 'vae_z0', 'vae_z1', 'vae_z2', 'vae_z3', 'vae_z4']
✅ Student v2 XGBoost loaded
✅ Student v2 LightGBM loaded (2000 trees)

Student v2 feature matrix: (9841, 71)
  Populated features: 3/71
  Student XGB probs: mean=0.0190, std=0.0111
  Student LGB probs: mean=0.0729, std=0.0542
  Student ensemble probs: mean=0.0052, std=0.0017

  XBlock Phishing Dataset — Model Comparison (N=9,841, fraud rate=22.1%)
Model                           ROC-AUC   PR-AUC       F1  Best F1   Best θ Feats Used
──────────────────────────────────────────────────────────────────────────────────────────
TX-Level Ensemble (prod)         0.6024   0.3710   0.0000   0.3953   0.0400       9/21
Student v2 XGB (addr)            0.6507   0.3869   0.0000   0.4665   0.0300       3/

---
## Dataset 3: Forta Network — Live Security Bot Labels

**Source**: Forta Foundation (100+ security researcher bots)  
**Labels**: Real-time detections: scam, phish, exploit, wash-trading  
**API**: Free, no auth required for basic queries  
**Why**: Live production labels from the largest blockchain security network

In [9]:
# ============================================================================
# CELL 7: Forta Network Label Validation
# ============================================================================
print('📊 Querying Forta Network for independently-labeled addresses...')
t0 = time.time()

# Forta Labels API (GraphQL)
FORTA_API = 'https://api.forta.network/labels/v1'

def query_forta_labels(label_type: str, limit: int = 500) -> list:
    """Query Forta API for addresses with a specific label."""
    params = {
        'labels': label_type,
        'chainId': 1,  # Ethereum mainnet
        'limit': limit,
    }
    try:
        r = requests.get(FORTA_API, params=params, timeout=30)
        if r.status_code == 200:
            data = r.json()
            if 'events' in data:
                return data['events']
            elif isinstance(data, list):
                return data
            else:
                return []
        else:
            print(f'   Forta API returned {r.status_code}: {r.text[:200]}')
            return []
    except Exception as e:
        print(f'   Forta API error: {e}')
        return []

# Query for known scam/phishing labels
forta_labels = {}
for label_type in ['scam', 'phishing', 'exploit', 'attack']:
    print(f'  Querying "{label_type}"...')
    events = query_forta_labels(label_type, limit=500)
    if events:
        addresses = set()
        for event in events:
            # Extract address from event
            if isinstance(event, dict):
                addr = event.get('entity', event.get('address', ''))
                if addr and addr.startswith('0x') and len(addr) == 42:
                    addresses.add(addr.lower())
        forta_labels[label_type] = addresses
        print(f'   → {len(addresses)} unique addresses')
    else:
        print(f'   → No results (API may require auth token)')

total_forta = set()
for addrs in forta_labels.values():
    total_forta.update(addrs)

print(f'\n  Total unique Forta-labeled addresses: {len(total_forta):,}')

if len(total_forta) > 0:
    # Cross-check: Load our training data addresses and see overlap
    try:
        df_our = pd.read_parquet(
            r'c:\amttp\processed\eth_transactions_full_labeled.parquet',
            columns=['from_address', 'to_address', 'fraud', 'value_eth',
                     'gas_price_gwei', 'gas_used', 'gas_limit', 'nonce', 'transaction_type',
                     'sender_total_transactions', 'sender_total_sent', 'sender_total_received',
                     'sender_balance', 'sender_avg_sent', 'sender_unique_receivers',
                     'sender_in_out_ratio', 'sender_active_duration_mins']
        )
        our_addresses = set(df_our['from_address'].str.lower().unique())
        
        # Find overlap
        overlap = total_forta & our_addresses
        print(f'\n  ── Forta ↔ Our Dataset Overlap ──')
        print(f'  Forta addresses: {len(total_forta):,}')
        print(f'  Our addresses:   {len(our_addresses):,}')
        print(f'  Overlap:         {len(overlap):,}')
        
        if len(overlap) > 10:
            # Get our model's predictions for overlapping addresses
            df_overlap = df_our[df_our['from_address'].str.lower().isin(overlap)].copy()
            
            # These are ALL confirmed scam/phish per Forta → ground truth = 1
            y_forta = np.ones(len(df_overlap), dtype=int)
            
            X_forta = build_feature_matrix(df_overlap)
            prob_forta = predict_ensemble(X_forta)
            
            # Detection rate: how many Forta-flagged addresses do we also flag?
            detected = (prob_forta >= OPTIMAL_THRESHOLD).sum()
            detection_rate = detected / len(prob_forta)
            
            print(f'\n  ── Our Model vs Forta Labels ──')
            print(f'  Forta-labeled transactions in our data: {len(df_overlap):,}')
            print(f'  Our model detects: {detected:,} / {len(prob_forta):,} ({detection_rate:.2%})')
            print(f'  Mean fraud probability: {prob_forta.mean():.4f}')
            print(f'  Our original fraud labels for these: {df_overlap["fraud"].mean():.2%}')
            
            # Agreement rate
            our_fraud = df_overlap['fraud'].values
            agreement = (our_fraud == 1).mean()
            print(f'  Agreement (our labels vs Forta): {agreement:.2%}')
            
            ALL_RESULTS.append({
                'name': 'Forta Network', 'n_samples': len(df_overlap),
                'n_pos': int(len(df_overlap)), 'detection_rate': round(detection_rate, 4),
                'mean_prob': round(float(prob_forta.mean()), 4),
                'agreement_with_our_labels': round(agreement, 4),
            })
        else:
            print(f'  Too few overlapping addresses ({len(overlap)}) for meaningful evaluation')
            
            ALL_RESULTS.append({
                'name': 'Forta Network', 'n_samples': 0,
                'note': f'Only {len(overlap)} overlapping addresses found'
            })
    except FileNotFoundError:
        print('  ⚠️  Training dataset not found locally (expected in Colab).')
        print('  Upload eth_transactions_full_labeled.parquet for Forta cross-check.')
else:
    print('  ⚠️  No Forta labels retrieved. API may require authentication.')
    print('  Get a free API key at: https://docs.forta.network/en/latest/api-keys/')
    
    ALL_RESULTS.append({
        'name': 'Forta Network', 'n_samples': 0,
        'note': 'API returned no results - may need auth token'
    })

print(f'\n  ⏱️  Completed in {time.time()-t0:.1f}s')

📊 Querying Forta Network for independently-labeled addresses...
  Querying "scam"...
   Forta API returned 404: 404 page not found

   → No results (API may require auth token)
  Querying "phishing"...
   Forta API returned 404: 404 page not found

   → No results (API may require auth token)
  Querying "exploit"...
   Forta API returned 404: 404 page not found

   → No results (API may require auth token)
  Querying "attack"...
   Forta API returned 404: 404 page not found

   → No results (API may require auth token)

  Total unique Forta-labeled addresses: 0
  ⚠️  No Forta labels retrieved. API may require authentication.
  Get a free API key at: https://docs.forta.network/en/latest/api-keys/

  ⏱️  Completed in 3.4s


---
## Dataset 4: OFAC SDN — U.S. Treasury Sanctioned Addresses

**Source**: Office of Foreign Assets Control (U.S. Treasury)  
**Labels**: Legally sanctioned addresses (Tornado Cash, Lazarus Group, ransomware)  
**Why**: Zero-ambiguity ground truth — these are officially designated illicit

In [10]:
# ============================================================================
# CELL 8: OFAC SDN Sanctioned Address Validation
# ============================================================================
print('📊 Checking OFAC SDN Sanctioned Ethereum Addresses...')
t0 = time.time()

# Known OFAC-sanctioned Ethereum addresses (from public Treasury announcements)
# Source: https://home.treasury.gov/policy-issues/financial-sanctions/specially-designated-nationals-and-blocked-persons-list-sdn-human-readable-lists
# These are publicly known sanctioned crypto addresses
OFAC_SANCTIONED = [
    # Tornado Cash (sanctioned Aug 2022)
    '0xd90e2f925da726b50c4ed8d0fb90ad053324f31b',
    '0xd96f2b1c14db8458374d9aca76e26c3d18364307',
    '0x4736dcf1b7a3d580672cce6e7c65cd5cc9cfbf68',
    '0xdd4c48c0b24039969fc16d1cdf626eab821d3384',
    '0xd4b88df4d29f5cedd6857912842cff3b20c8cfa3',
    '0x910cb14c8ae0a3a15a26dcc6d66c39458b878e79',
    '0xa160cdab225685da1d56aa342ad8841c3b53f291',
    '0xfd8610d20aa15b7b2e3be39b396a1bc3516c7144',
    '0xf60dd140cff0706bae9cd734ac3683696c17a0b3',
    '0x22aaa7720ddd5388a3c0a3333430953c68f1849b',
    '0xba214c1c1928a32bffe790263e38b4af9bfcd659',
    '0xb1c8094b234dce6e03f10a5b673c1d8c69739a00',
    '0x527653ea119f3e6a1f5bd18fbf4714081d7b31ce',
    '0x58e8dcc13be9780fc42e8723d8ead4cf46943df2',
    '0xd691f27f38b395864ea86cfc7253969b409c362d',
    '0xaeaac358560e11f52454d997aaff2c5731b6f8a6',
    '0x1356c899d8c9467c7f71c195612f8a395abf2f0a',
    '0xa7e5d5a720f06526557c513402f2e6b5fa20b008',
    '0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc',
    '0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936',
    '0x23773e65ed146a459791799d01336db287f25334',
    '0xd21be7248e0197ee08e0c20d4a398dad3e6e232c',
    '0x610b717796ad172b316836ac95a2ffad065ceab4',
    '0x178169b423a011fff22b9e3f3abea13414ddd0f1',
    '0xbb93e510bbcd0b7beb5a853875f9ec60275cf498',
    # Lazarus Group / DPRK (various sanctions)
    '0x098b716b8aaf21512996dc57eb0615e2383e2f96',
    '0xa0e1c89ef1a489c9c7de96311ed5ce5d32c20e4b',
    '0x3cffd56b47b7b41c56258d9c7731abadc360e460',
    '0x53b6936513e738f44fb50d2b9476730c0ab3bfc1',
    # Blender.io (sanctioned May 2022)
    '0x8589427373d6d84e98730d7795d8f6f8731fda16',
    '0x722122df12d4e14e13ac3b6895a86e84145b6967',
    '0xca0840578f57fe216fab5aab07c7379c83067836',
]

# Also try to fetch the latest list from OFAC API
print(f'  Known sanctioned addresses: {len(OFAC_SANCTIONED)}')

# Try to download the full SDN list and extract crypto addresses
OFAC_SDN_URL = 'https://www.treasury.gov/ofac/downloads/sdn.csv'
try:
    print('  Fetching latest OFAC SDN list...')
    r = requests.get(OFAC_SDN_URL, timeout=30)
    if r.status_code == 200:
        import re
        # Extract Ethereum addresses from the SDN text
        eth_pattern = re.compile(r'0x[a-fA-F0-9]{40}')
        additional = set(addr.lower() for addr in eth_pattern.findall(r.text))
        existing = set(a.lower() for a in OFAC_SANCTIONED)
        new_addrs = additional - existing
        if new_addrs:
            OFAC_SANCTIONED.extend(list(new_addrs))
            print(f'  Found {len(new_addrs)} additional addresses from live SDN list')
        print(f'  Total sanctioned addresses: {len(OFAC_SANCTIONED)}')
    else:
        print(f'  SDN download returned {r.status_code}, using hardcoded list')
except Exception as e:
    print(f'  SDN download failed ({e}), using hardcoded list')

# Cross-check against our dataset
try:
    df_our = pd.read_parquet(
        r'c:\amttp\processed\eth_transactions_full_labeled.parquet',
        columns=['from_address', 'to_address', 'fraud', 'value_eth',
                 'gas_price_gwei', 'gas_used', 'gas_limit', 'nonce', 'transaction_type',
                 'sender_total_transactions', 'sender_total_sent', 'sender_total_received',
                 'sender_balance', 'sender_avg_sent', 'sender_unique_receivers',
                 'sender_in_out_ratio', 'sender_active_duration_mins']
    )
    
    ofac_set = set(a.lower() for a in OFAC_SANCTIONED)
    our_addresses = set(df_our['from_address'].str.lower().unique())
    
    # Check overlap
    overlap = ofac_set & our_addresses
    print(f'\n  ── OFAC ↔ Our Dataset ──')
    print(f'  OFAC sanctioned: {len(ofac_set)}')
    print(f'  Our addresses:   {len(our_addresses):,}')
    print(f'  Overlap:         {len(overlap)}')
    
    if len(overlap) > 0:
        df_ofac = df_our[df_our['from_address'].str.lower().isin(overlap)].copy()
        
        X_ofac = build_feature_matrix(df_ofac)
        prob_ofac = predict_ensemble(X_ofac)
        
        detected = (prob_ofac >= OPTIMAL_THRESHOLD).sum()
        detection_rate = detected / len(prob_ofac)
        
        print(f'\n  ── Our Model vs OFAC Ground Truth ──')
        print(f'  Sanctioned address transactions: {len(df_ofac):,}')
        print(f'  Our model detects: {detected:,} / {len(prob_ofac):,} ({detection_rate:.2%})')
        print(f'  Mean fraud probability: {prob_ofac.mean():.4f}')
        print(f'  Our original labels: {df_ofac["fraud"].mean():.2%} fraud')
        
        # Per-address breakdown
        print(f'\n  Per-address detection:')
        for addr in list(overlap)[:10]:
            mask = df_ofac['from_address'].str.lower() == addr
            if mask.sum() > 0:
                addr_probs = prob_ofac[mask.values]
                flagged = (addr_probs >= OPTIMAL_THRESHOLD).sum()
                our_label = df_ofac.loc[mask, 'fraud'].values[0]
                print(f'    {addr[:10]}...{addr[-4:]}  prob={addr_probs.mean():.4f}  '
                      f'flagged={flagged}/{len(addr_probs)}  our_label={our_label}')
        
        ALL_RESULTS.append({
            'name': 'OFAC SDN', 'n_samples': len(df_ofac),
            'n_pos': len(df_ofac),  # All sanctioned = positive
            'detection_rate': round(detection_rate, 4),
            'mean_prob': round(float(prob_ofac.mean()), 4),
            'agreement_with_our_labels': round(float(df_ofac['fraud'].mean()), 4),
        })
    else:
        print('  No overlap — OFAC addresses not in our 30-day BigQuery dataset')
        print('  This is expected: our data is a 30-day snapshot, OFAC addresses')
        print('  may not have transacted during that window.')
        
        # Alternative: Score the OFAC addresses with zero features
        # (simulates a cold-start scenario)
        print(f'\n  ── Cold-Start Scoring (zero sender features) ──')
        dummy = pd.DataFrame({'value_eth': [0.0] * len(OFAC_SANCTIONED)})
        X_dummy = build_feature_matrix(dummy)
        prob_dummy = predict_ensemble(X_dummy)
        
        print(f'  Mean fraud prob (zero features): {prob_dummy.mean():.4f}')
        print(f'  This is our model\'s base rate for unknown addresses')
        
        ALL_RESULTS.append({
            'name': 'OFAC SDN', 'n_samples': 0,
            'note': 'No overlap with our dataset - addresses not in 30-day window',
            'cold_start_mean_prob': round(float(prob_dummy.mean()), 4),
        })

except FileNotFoundError:
    print('  ⚠️  Training dataset not found locally. Upload for overlap check.')

print(f'\n  ⏱️  Completed in {time.time()-t0:.1f}s')

📊 Checking OFAC SDN Sanctioned Ethereum Addresses...
  Known sanctioned addresses: 32
  Fetching latest OFAC SDN list...
  Found 53 additional addresses from live SDN list
  Total sanctioned addresses: 85

  ── OFAC ↔ Our Dataset ──
  OFAC sanctioned: 85
  Our addresses:   550,226
  Overlap:         0
  No overlap — OFAC addresses not in our 30-day BigQuery dataset
  This is expected: our data is a 30-day snapshot, OFAC addresses
  may not have transacted during that window.

  ── Cold-Start Scoring (zero sender features) ──
  Mean fraud prob (zero features): 0.0670
  This is our model's base rate for unknown addresses

  ⏱️  Completed in 17.6s


---
## Dataset 5: Chainabuse — Community-Reported Scam Addresses

**Source**: Chainabuse.com (community reports)  
**Labels**: User-reported scam, phishing, rug pull addresses  
**Why**: Large-scale community intelligence, different source than our training pipeline

In [11]:
# ============================================================================
# CELL 9: Chainabuse Community Reports
# ============================================================================
print('📊 Checking Chainabuse community-reported scam addresses...')
t0 = time.time()

# Chainabuse API (GraphQL)
CHAINABUSE_API = 'https://www.chainabuse.com/api/graphql-proxy'

# Well-known scam addresses from Chainabuse (curated subset)
# These are frequently reported on Chainabuse and other sources
CHAINABUSE_KNOWN_SCAMS = [
    # Pig butchering / romance scams (frequently reported)
    '0x7bd9902e5d04e4e0a628edcb8cdec6dbc5a9f115',
    '0x3e0e0e6f3e0e3e0e3e0e3e0e3e0e3e0e3e0e3e0e',  # Placeholder
    # Known rug pulls
    '0x4b6aab87c338fa4e66bd7e42c27ca43d440bc829',
    '0x6b3595068778dd592e39a122f4f5a5cf09c90fe2',  # SushiSwap deployer (not scam, control)
    # Known phishing contracts
    '0xd882cfc20f52f2599d84b8e8d58c7fb62cfe344b',
    '0x2b591e99afe9f32eaa6214f7b7629768c40eeb39',  # HEX (controversial, control)
]

# Try Chainabuse API
chainabuse_addresses = set()
try:
    # Chainabuse GraphQL query for recent Ethereum scam reports
    query = {
        'query': '''
            query {
                getReports(input: { chain: "ethereum", limit: 100 }) {
                    reports {
                        address
                        category
                        description
                    }
                }
            }
        '''
    }
    r = requests.post(CHAINABUSE_API, json=query, timeout=15,
                      headers={'Content-Type': 'application/json'})
    if r.status_code == 200:
        data = r.json()
        reports = data.get('data', {}).get('getReports', {}).get('reports', [])
        for report in reports:
            addr = report.get('address', '')
            if addr.startswith('0x') and len(addr) == 42:
                chainabuse_addresses.add(addr.lower())
        print(f'  Chainabuse API: {len(chainabuse_addresses)} addresses from recent reports')
    else:
        print(f'  Chainabuse API returned {r.status_code} (may need auth or have rate limits)')
except Exception as e:
    print(f'  Chainabuse API error: {e}')

# Combine with known addresses
chainabuse_addresses.update(a.lower() for a in CHAINABUSE_KNOWN_SCAMS if a.startswith('0x') and len(a) == 42)
print(f'  Total Chainabuse addresses to check: {len(chainabuse_addresses)}')

if len(chainabuse_addresses) > 0:
    try:
        df_our = pd.read_parquet(
            r'c:\amttp\processed\eth_transactions_full_labeled.parquet',
            columns=['from_address', 'fraud', 'value_eth',
                     'gas_price_gwei', 'gas_used', 'gas_limit', 'nonce', 'transaction_type',
                     'sender_total_transactions', 'sender_total_sent', 'sender_total_received',
                     'sender_balance', 'sender_avg_sent', 'sender_unique_receivers',
                     'sender_in_out_ratio', 'sender_active_duration_mins']
        )
        
        overlap = chainabuse_addresses & set(df_our['from_address'].str.lower().unique())
        print(f'  Overlap with our dataset: {len(overlap)}')
        
        if len(overlap) > 0:
            df_ca = df_our[df_our['from_address'].str.lower().isin(overlap)].copy()
            X_ca = build_feature_matrix(df_ca)
            prob_ca = predict_ensemble(X_ca)
            
            detected = (prob_ca >= OPTIMAL_THRESHOLD).sum()
            print(f'\n  ── Our Model vs Chainabuse Reports ──')
            print(f'  Transactions from reported addresses: {len(df_ca):,}')
            print(f'  Detected as fraud: {detected:,} ({detected/len(df_ca):.2%})')
            print(f'  Mean fraud prob: {prob_ca.mean():.4f}')
            print(f'  Our original labels: {df_ca["fraud"].mean():.2%} fraud')
            
            ALL_RESULTS.append({
                'name': 'Chainabuse', 'n_samples': len(df_ca),
                'n_pos': len(df_ca),
                'detection_rate': round(detected/len(df_ca), 4),
                'mean_prob': round(float(prob_ca.mean()), 4),
            })
        else:
            print('  No overlap with our 30-day dataset')
            ALL_RESULTS.append({'name': 'Chainabuse', 'n_samples': 0, 'note': 'No overlap'})
    except FileNotFoundError:
        print('  ⚠️  Training dataset not found locally.')

print(f'\n  ⏱️  Completed in {time.time()-t0:.1f}s')

📊 Checking Chainabuse community-reported scam addresses...
  Chainabuse API returned 400 (may need auth or have rate limits)
  Total Chainabuse addresses to check: 6
  Overlap with our dataset: 0
  No overlap with our 30-day dataset

  ⏱️  Completed in 4.7s


---
## Summary Report

In [12]:
# ============================================================================
# CELL 10: Final Summary Report
# ============================================================================
print('\n' + '=' * 80)
print('           CROSS-VALIDATION SUMMARY REPORT')
print('           AMTTP TX-Level Ensemble vs External Ground Truth')
print('=' * 80)

print(f'\nDate: {datetime.now().isoformat()}')
print(f'Model: TX-Level Ensemble (XGB + LGB + Meta-Learner)')
print(f'Production threshold: {OPTIMAL_THRESHOLD:.4f}')
print(f'Training ROC-AUC: 0.9879 (on weakly-supervised labels)')

print(f'\n{"─"*80}')
print(f'{"Dataset":<25} {"N":>8} {"ROC-AUC":>10} {"PR-AUC":>10} {"F1":>8} {"Detect%":>10} {"Notes":<20}')
print(f'{"─"*80}')

for r in ALL_RESULTS:
    name = r.get('name', '?')
    n = r.get('n_samples', 0)
    roc = r.get('roc_auc', '')
    pr = r.get('pr_auc', '')
    f1 = r.get('f1_production', '')
    detect = r.get('detection_rate', '')
    note = r.get('note', r.get('error', ''))
    
    roc_str = f'{roc:.4f}' if isinstance(roc, float) else str(roc)
    pr_str = f'{pr:.4f}' if isinstance(pr, float) else str(pr)
    f1_str = f'{f1:.4f}' if isinstance(f1, float) else str(f1)
    det_str = f'{detect:.2%}' if isinstance(detect, float) else str(detect)
    
    print(f'{name:<25} {n:>8,} {roc_str:>10} {pr_str:>10} {f1_str:>8} {det_str:>10} {note:<20}')

print(f'{"─"*80}')

# Reference: our training performance
print(f'\n{"Training (reference)":<25} {"1.67M":>8} {"0.9879":>10} {"0.9715":>10} {"0.9012":>8} {"87.65%":>10} {"weak supervision":<20}')

print(f'\n{"="*80}')
print('INTERPRETATION GUIDE')
print(f'{"="*80}')
print('''
  Elliptic (feature-mapped):
    - Features are anonymised and cross-chain (Bitcoin → Ethereum)
    - ROC-AUC > 0.65 = meaningful cross-domain fraud signal
    - ROC-AUC > 0.75 = strong generalisation
    - The "concept alignment test" is more informative than raw metrics
    
  XBlock Phishing:
    - Address-level features only (no tx-level, no gas)
    - Tests whether sender aggregate features alone detect phishing
    - ROC-AUC > 0.60 = our sender features capture phishing patterns

  Forta Network:
    - Detection rate matters more than ROC-AUC (all positive class)
    - Detection > 50% = strong alignment with security researchers
    - Agreement with our labels validates our labeling pipeline

  OFAC SDN:
    - Binary test: do we flag sanctioned addresses?
    - Detection > 80% = regulatory compliance ready
    - Low overlap expected (30-day window vs historical sanctions)

  Chainabuse:
    - Community reports (noisier labels)
    - Detection > 40% = reasonable alignment with crowd intelligence
''')

# Save results
results_file = Path(DATA_DIR) / 'cross_validation_results.json'
with open(results_file, 'w') as f:
    json.dump({
        'date': datetime.now().isoformat(),
        'model': 'tx-level-ensemble-v1.0',
        'threshold': OPTIMAL_THRESHOLD,
        'results': ALL_RESULTS,
    }, f, indent=2, default=str)
print(f'\n📁 Results saved to: {results_file}')


           CROSS-VALIDATION SUMMARY REPORT
           AMTTP TX-Level Ensemble vs External Ground Truth

Date: 2026-02-10T00:28:25.674631
Model: TX-Level Ensemble (XGB + LGB + Meta-Learner)
Production threshold: 0.5950
Training ROC-AUC: 0.9879 (on weakly-supervised labels)

────────────────────────────────────────────────────────────────────────────────
Dataset                          N    ROC-AUC     PR-AUC       F1    Detect% Notes               
────────────────────────────────────────────────────────────────────────────────
Elliptic (feature-mapped)   46,564     0.3484     0.0688   0.0025                                
XBlock Phishing              9,841     0.6024     0.3710   0.0000                                
Forta Network                    0                                           API returned no results - may need auth token
OFAC SDN                         0                                           No overlap with our dataset - addresses not in 30-day window
Chainabu

## Full Risk Engine Stack Comparison

Test **every model generation** (Teacher, Student v2, TX-Level) × (XGB, LGB, Ensemble)  
plus **rule-based / graph pattern detection** adapted to XBlock address aggregates,  
and **combined ML + Rules** to measure detection coverage uplift.

## Feature Reconstruction — Graph Properties & FATF Pattern Scores

Reconstruct features the same way the training pipeline does:
1. **7 FATF Pattern Scores** (from `sophisticated_fraud_detection_ultra.py`): SMURFING, FAN_OUT, FAN_IN, LAYERING, STRUCTURING, VELOCITY, PEELING
2. **`sender_sophisticated_score`** = sum of all pattern scores
3. **`sender_hybrid_score`** = 0.4×XGB_normalized + 0.3×pattern_boost + 0.3×soph_normalized
4. **Graph properties** approximated from address aggregates: degree, in/out ratio, degree centrality
5. **Properly-aligned TX-Level features** (14/21 instead of 9/21)

In [10]:
# ============================================================================
# FEATURE RECONSTRUCTION — Replicate training pipeline from XBlock aggregates
# ============================================================================
# Source formulas: scripts/sophisticated_fraud_detection_ultra.py
# Source pipeline: scripts/create_complete_labeled_dataset.py
# ============================================================================

import xgboost as xgb
import numpy as np
import pandas as pd
import json
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

print('='*90)
print('  FEATURE RECONSTRUCTION — Replicating Training Pipeline on XBlock')
print('='*90)

# ── Extract raw XBlock columns as numeric ──
sent_tnx     = pd.to_numeric(df_xb['Sent tnx'], errors='coerce').fillna(0).values
recv_tnx     = pd.to_numeric(df_xb['Received Tnx'], errors='coerce').fillna(0).values
unique_sent  = pd.to_numeric(df_xb['Unique Sent To Addresses'], errors='coerce').fillna(0).values
unique_recv  = pd.to_numeric(df_xb['Unique Received From Addresses'], errors='coerce').fillna(0).values
total_sent   = pd.to_numeric(df_xb['total Ether sent'], errors='coerce').fillna(0).values
total_recv   = pd.to_numeric(df_xb['total ether received'], errors='coerce').fillna(0).values
balance      = pd.to_numeric(df_xb['total ether balance'], errors='coerce').fillna(0).values
avg_sent     = pd.to_numeric(df_xb['avg val sent'], errors='coerce').fillna(0).values
avg_recv     = pd.to_numeric(df_xb['avg val received'], errors='coerce').fillna(0).values
max_sent     = pd.to_numeric(df_xb['max val sent'], errors='coerce').fillna(0).values
min_sent     = pd.to_numeric(df_xb['min val sent'], errors='coerce').fillna(0).values
max_recv     = pd.to_numeric(df_xb['max value received '], errors='coerce').fillna(0).values
min_recv     = pd.to_numeric(df_xb['min value received'], errors='coerce').fillna(0).values
n_contracts  = pd.to_numeric(df_xb['Number of Created Contracts'], errors='coerce').fillna(0).values
time_diff    = pd.to_numeric(df_xb['Time Diff between first and last (Mins)'], errors='coerce').fillna(0).values
avg_min_sent = pd.to_numeric(df_xb['Avg min between sent tnx'], errors='coerce').fillna(0).values
avg_min_recv = pd.to_numeric(df_xb['Avg min between received tnx'], errors='coerce').fillna(0).values

# ERC20 columns
erc20_total_recv   = pd.to_numeric(df_xb.get('ERC20 total Ether received', pd.Series(0, index=df_xb.index)), errors='coerce').fillna(0).values
erc20_total_sent   = pd.to_numeric(df_xb.get('ERC20 total ether sent', pd.Series(0, index=df_xb.index)), errors='coerce').fillna(0).values
erc20_uniq_sent    = pd.to_numeric(df_xb.get('ERC20 uniq sent addr', pd.Series(0, index=df_xb.index)), errors='coerce').fillna(0).values
erc20_uniq_recv    = pd.to_numeric(df_xb.get('ERC20 uniq rec addr', pd.Series(0, index=df_xb.index)), errors='coerce').fillna(0).values
erc20_tnxs         = pd.to_numeric(df_xb.get('Total ERC20 tnxs', pd.Series(0, index=df_xb.index)), errors='coerce').fillna(0).values

N = len(df_xb)
print(f'\n  XBlock records: {N:,}')

# ============================================================================
# PART 0: Load and run Teacher XGB on XBlock
# (The ACTUAL model used in the training pipeline to score addresses)
# Source: sophisticated_fraud_detection_ultra.py XGB cross-validation section
# ============================================================================
print('\n── Loading Teacher XGB (Gen 1) for hybrid_score computation ──')

teacher_dir = Path(r'c:\amttp\ml\Automation\ml_pipeline\models\trained')
teacher_schema_path = Path(r'c:\amttp\ml\Automation\ml_pipeline\models\feature_schema.json')

with open(teacher_schema_path) as f:
    teacher_schema = json.load(f)
teacher_feature_names = teacher_schema['feature_names']  # 171 features

teacher_xgb_model = xgb.XGBClassifier()
teacher_xgb_model.load_model(str(teacher_dir / 'xgb.json'))
print(f'  Loaded Teacher XGB ({len(teacher_feature_names)} features)')

# Map XBlock columns → Teacher's 171-feature schema
# This is the same mapping used in the original Kaggle dataset since Teacher was trained on it
teacher_xb_mapping = {
    'avg_min_between_received_tnx': 'Avg min between received tnx',
    'avg_min_between_sent_tnx':     'Avg min between sent tnx',
    'avg_val_received':             'avg val received',
    'avg_val_sent':                 'avg val sent',
    'max_val_sent':                 'max val sent',
    'max_value_received':           'max value received ',
    'min_val_sent':                 'min val sent',
    'min_value_received':           'min value received',
    'number_of_created_contracts':  'Number of Created Contracts',
    'received_tnx':                 'Received Tnx',
    'sent_tnx':                     'Sent tnx',
    'time_diff_between_first_and_last_(mins)': 'Time Diff between first and last (Mins)',
    'total_erc20_tnxs':             'Total ERC20 tnxs',
    'total_ether_balance':          'total ether balance',
    'total_ether_received':         'total ether received',
    'total_ether_sent':             'total Ether sent',
    'unique_received_from_addresses': 'Unique Received From Addresses',
    'unique_sent_to_addresses':     'Unique Sent To Addresses',
    'erc20_total_ether_received':   'ERC20 total Ether received',
    'erc20_total_ether_sent':       'ERC20 total ether sent',
    'erc20_uniq_sent_addr':         'ERC20 uniq sent addr',
    'erc20_uniq_rec_addr':          'ERC20 uniq rec addr',
    'erc20_avg_val_rec':            'ERC20 avg val rec',
    'erc20_avg_val_sent':           'ERC20 avg val sent',
    'erc20_min_val_rec':            'ERC20 min val rec',
    'erc20_min_val_sent':           'ERC20 min val sent',
    'erc20_max_val_rec':            'ERC20 max val rec',
    'erc20_max_val_sent':           'ERC20 max val sent',
    'erc20_uniq_rec_contract_addr': 'ERC20 uniq rec contract addr',
    'erc20_uniq_sent_token_name':   'ERC20 uniq sent token name',
    'erc20_uniq_rec_token_name':    'ERC20 uniq rec token name',
}

X_teacher = np.zeros((N, len(teacher_feature_names)), dtype=np.float32)
teacher_mapped_count = 0
for i, feat in enumerate(teacher_feature_names):
    if feat in teacher_xb_mapping:
        xb_col = teacher_xb_mapping[feat]
        if xb_col in df_xb.columns:
            vals = pd.to_numeric(df_xb[xb_col], errors='coerce').fillna(0).values
            X_teacher[:, i] = vals
            if (vals != 0).any():
                teacher_mapped_count += 1

# Set _was_missing indicators for unmapped features
for i, feat in enumerate(teacher_feature_names):
    if feat.endswith('_was_missing'):
        base_feat = feat.replace('_was_missing', '')
        if base_feat not in teacher_xb_mapping:
            X_teacher[:, i] = 1.0

X_teacher = np.nan_to_num(X_teacher, nan=0.0, posinf=0.0, neginf=0.0)
print(f'  Features populated: {teacher_mapped_count}/{len(teacher_schema["core_features"])} core features')

# Get ACTUAL Teacher XGB predictions (raw probabilities)
teacher_xgb_raw = teacher_xgb_model.predict_proba(X_teacher)[:, 1]
print(f'  Teacher XGB raw probs: mean={teacher_xgb_raw.mean():.6f}, std={teacher_xgb_raw.std():.6f}, '
      f'min={teacher_xgb_raw.min():.6f}, max={teacher_xgb_raw.max():.6f}')

# Normalize to 0-100 scale (same as sophisticated_fraud_detection_ultra.py)
# xgb_normalized = (xgb_raw / max(xgb_raw)) * 100
max_raw = teacher_xgb_raw.max()
xgb_normalized = (teacher_xgb_raw / max_raw * 100) if max_raw > 0 else np.zeros(N)
print(f'  XGB normalized (0-100): mean={xgb_normalized.mean():.2f}, std={xgb_normalized.std():.2f}')

# ============================================================================
# PART 1: Compute 7 FATF Pattern Scores
# (Exact formulas from sophisticated_fraud_detection_ultra.py)
# ============================================================================
print('\n── Computing 7 FATF Pattern Scores ──')

# Constants from the training pipeline
SMURFING_THRESHOLD = 1.0
FAN_THRESHOLD = 10

# --- 1. SMURFING: many small txs to many recipients ---
smurf_mask = (sent_tnx >= 5) & (avg_sent < SMURFING_THRESHOLD) & (total_sent > 5) & (unique_sent >= 3)
smurf_score = np.where(smurf_mask,
    np.clip(sent_tnx / 10, 0, 1) * 30 +
    np.clip(1 - avg_sent / SMURFING_THRESHOLD, 0, 1) * 25 +
    np.clip(unique_sent / 10, 0, 1) * 20 +
    np.clip(total_sent / 50, 0, 1) * 25,
    0.0
)
smurf_score = np.where(smurf_score > 30, smurf_score, 0.0)
print(f'  SMURFING:    {(smurf_score > 0).sum():>5} addresses flagged' if (smurf_score>0).any() else '  SMURFING:        0 addresses flagged')

# --- 2. FAN_OUT ---
fan_out_mask = (unique_sent >= FAN_THRESHOLD) & (sent_tnx >= FAN_THRESHOLD)
fan_out_score = np.where(fan_out_mask,
    np.clip(unique_sent / 20, 0, 1) * 40 +
    np.clip(sent_tnx / 20, 0, 1) * 30 +
    np.clip(total_sent / 100, 0, 1) * 30,
    0.0
)
print(f'  FAN_OUT:     {(fan_out_score > 0).sum():>5} addresses flagged' if (fan_out_score>0).any() else '  FAN_OUT:         0 addresses flagged')

# --- 3. FAN_IN ---
fan_in_mask = (unique_recv >= FAN_THRESHOLD) & (recv_tnx >= FAN_THRESHOLD)
fan_in_score = np.where(fan_in_mask,
    np.clip(unique_recv / 20, 0, 1) * 40 +
    np.clip(recv_tnx / 20, 0, 1) * 30 +
    np.clip(total_recv / 100, 0, 1) * 30,
    0.0
)
print(f'  FAN_IN:      {(fan_in_score > 0).sum():>5} addresses flagged' if (fan_in_score>0).any() else '  FAN_IN:          0 addresses flagged')

# --- 4. LAYERING ---
pass_ratio = np.clip(total_sent / np.maximum(total_recv, 0.001), 0, 1)
layer_mask = (pass_ratio > 0.8) & (recv_tnx >= 2) & (sent_tnx >= 2)
layering_score = np.where(layer_mask,
    pass_ratio * 40 +
    np.clip(recv_tnx / 5, 0, 1) * 30 +
    np.clip(sent_tnx / 5, 0, 1) * 30,
    0.0
)
print(f'  LAYERING:    {(layering_score > 0).sum():>5} addresses flagged' if (layering_score>0).any() else '  LAYERING:        0 addresses flagged')

# --- 5. STRUCTURING ---
round_amounts = np.array([0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0, 1000.0])
is_near_round = np.zeros(N, dtype=bool)
for ra in round_amounts:
    is_near_round |= (np.abs(avg_sent - ra) / max(ra, 0.01) < 0.05)
is_near_round |= ((avg_sent >= 1) & (np.abs(avg_sent - np.floor(avg_sent)) < 0.01))

struct_mask = is_near_round & (sent_tnx >= 3)
approx_round_ratio = np.where(struct_mask, 0.7, 0.0)
structuring_score = np.where(struct_mask,
    approx_round_ratio * 40 +
    np.clip(sent_tnx / 10, 0, 1) * 30 +
    np.clip(total_sent / 50, 0, 1) * 30,
    0.0
)
print(f'  STRUCTURING: {(structuring_score > 0).sum():>5} addresses flagged' if (structuring_score>0).any() else '  STRUCTURING:     0 addresses flagged')

# --- 6. VELOCITY ---
days_active = time_diff / (24 * 60)
txs_per_day = np.where(days_active > 0, sent_tnx / days_active, sent_tnx)
velocity_mask = (txs_per_day > 20) & (sent_tnx >= 5)
velocity_score = np.where(velocity_mask,
    np.clip(txs_per_day / 100, 0, 1) * 50 +
    np.clip(sent_tnx / 20, 0, 1) * 50,
    0.0
)
print(f'  VELOCITY:    {(velocity_score > 0).sum():>5} addresses flagged' if (velocity_score>0).any() else '  VELOCITY:        0 addresses flagged')

# --- 7. PEELING ---
peel_ratio = np.where(max_sent > 0, 1 - (min_sent / np.maximum(max_sent, 0.001)), 0)
peel_mask = (sent_tnx >= 3) & (peel_ratio > 0.6) & (max_sent > 0)
peeling_score = np.where(peel_mask,
    np.clip(peel_ratio, 0, 1) * 50 +
    np.clip(sent_tnx / 10, 0, 1) * 50,
    0.0
)
print(f'  PEELING:     {(peeling_score > 0).sum():>5} addresses flagged' if (peeling_score>0).any() else '  PEELING:         0 addresses flagged')

# ============================================================================
# PART 2: Compute sender_sophisticated_score
# = sum of all pattern scores (same as training pipeline)
# ============================================================================
print('\n── Computing sender_sophisticated_score ──')

sophisticated_score = (
    smurf_score + fan_out_score + fan_in_score +
    layering_score + structuring_score + velocity_score + peeling_score
)

pattern_count = (
    (smurf_score > 0).astype(int) + (fan_out_score > 0).astype(int) +
    (fan_in_score > 0).astype(int) + (layering_score > 0).astype(int) +
    (structuring_score > 0).astype(int) + (velocity_score > 0).astype(int) +
    (peeling_score > 0).astype(int)
)

print(f'  Addresses with any pattern: {(sophisticated_score > 0).sum():,} ({(sophisticated_score > 0).mean():.1%})')
print(f'  Multi-pattern (≥2):         {(pattern_count >= 2).sum():,} ({(pattern_count >= 2).mean():.1%})')
print(f'  Score range: [{sophisticated_score.min():.1f}, {sophisticated_score.max():.1f}], mean={sophisticated_score[sophisticated_score>0].mean():.1f}' if (sophisticated_score>0).any() else '  No patterns detected')

# ============================================================================
# PART 3: Compute sender_hybrid_score using ACTUAL Teacher XGB predictions
# Formula: 0.4 * xgb_normalized + 0.3 * pattern_boost + 0.3 * soph_normalized
# Source: sophisticated_fraud_detection_ultra.py lines 662-668
# ============================================================================
print('\n── Computing sender_hybrid_score (with REAL Teacher XGB scores) ──')

# Pattern boost values (from PATTERN_BOOST dict in sophisticated_fraud_detection_ultra.py)
PATTERN_BOOST = {
    'smurf': 25, 'fan_out': 15, 'fan_in': 15,
    'layering': 15, 'structuring': 20, 'velocity': 15, 'peeling': 20
}

pattern_boost = (
    (smurf_score > 0).astype(float) * PATTERN_BOOST['smurf'] +
    (fan_out_score > 0).astype(float) * PATTERN_BOOST['fan_out'] +
    (fan_in_score > 0).astype(float) * PATTERN_BOOST['fan_in'] +
    (layering_score > 0).astype(float) * PATTERN_BOOST['layering'] +
    (structuring_score > 0).astype(float) * PATTERN_BOOST['structuring'] +
    (velocity_score > 0).astype(float) * PATTERN_BOOST['velocity'] +
    (peeling_score > 0).astype(float) * PATTERN_BOOST['peeling']
)
pattern_boost = np.clip(pattern_boost, 0, 100)  # Cap at 100

# Soph normalized: scale to 0-100
max_soph = sophisticated_score.max()
soph_normalized = (sophisticated_score / max_soph * 100) if max_soph > 0 else np.zeros(N)

# ACTUAL hybrid_score = 0.4 * xgb_normalized + 0.3 * pattern_boost + 0.3 * soph_normalized
# xgb_normalized comes from the REAL Teacher XGB predictions (computed in PART 0 above)
hybrid_score = (
    xgb_normalized * 0.4 +
    pattern_boost * 0.3 +
    soph_normalized * 0.3
)

print(f'  XGB normalized (REAL Teacher):  mean={xgb_normalized.mean():.2f}, range=[{xgb_normalized.min():.2f}, {xgb_normalized.max():.2f}]')
print(f'  Pattern boost:                  mean={pattern_boost.mean():.2f}, range=[{pattern_boost.min():.2f}, {pattern_boost.max():.2f}]')
print(f'  Soph normalized:                mean={soph_normalized.mean():.2f}, range=[{soph_normalized.min():.2f}, {soph_normalized.max():.2f}]')
print(f'  Hybrid score (REAL):            mean={hybrid_score.mean():.2f}, range=[{hybrid_score.min():.2f}, {hybrid_score.max():.2f}]')
print(f'  Addresses with hybrid > 0:      {(hybrid_score > 0).sum():,} ({(hybrid_score > 0).mean():.1%})')

# ============================================================================
# PART 4: Reconstruct graph-derived properties
# (Approximated from address aggregates since we don't have actual graph)
# Source: scripts/graph_analytics.py (Memgraph Cypher queries)
# ============================================================================
print('\n── Approximating Graph Properties ──')

total_transactions = sent_tnx + recv_tnx
unique_counterparties = unique_sent + unique_recv
in_out_ratio = np.where(sent_tnx > 0, recv_tnx / sent_tnx, recv_tnx)

degree = recv_tnx + sent_tnx
max_degree = degree.max() if degree.max() > 0 else 1
degree_centrality = degree / max_degree
weighted_degree = total_sent + total_recv
clustering_approx = np.where(
    total_transactions > 1,
    unique_counterparties / total_transactions,
    0.0
)
clustering_approx = np.clip(clustering_approx, 0, 1)

print(f'  Degree range: [{degree.min():.0f}, {degree.max():.0f}], mean={degree.mean():.1f}')
print(f'  Degree centrality range: [{degree_centrality.min():.4f}, {degree_centrality.max():.4f}]')

# ============================================================================
# PART 5: Build ENRICHED feature matrices for ALL model generations
# ============================================================================
print('\n── Building Enriched Feature Matrices ──')

# ── TX-Level (Gen 3): 21 features ──
TX_FEATURES = [
    'value_eth', 'gas_price_gwei', 'gas_used', 'gas_limit', 'nonce', 'transaction_type',
    'gas_efficiency', 'value_log', 'gas_price_log', 'is_round_amount',
    'value_gas_ratio', 'is_zero_value', 'is_high_nonce',
    'sender_total_transactions', 'sender_total_sent', 'sender_total_received',
    'sender_balance', 'sender_avg_sent', 'sender_unique_receivers',
    'sender_in_out_ratio', 'sender_active_duration_mins'
]

X_tx_enriched = np.zeros((N, 21), dtype=np.float32)
X_tx_enriched[:, 0] = avg_sent                                    # value_eth ≈ avg_val_sent
X_tx_enriched[:, 7] = np.log1p(avg_sent)                          # value_log = log1p(value_eth)
X_tx_enriched[:, 9] = is_near_round.astype(np.float32)            # is_round_amount
X_tx_enriched[:, 11] = (avg_sent == 0).astype(np.float32)         # is_zero_value
X_tx_enriched[:, 13] = sent_tnx + recv_tnx                        # sender_total_transactions
X_tx_enriched[:, 14] = total_sent                                  # sender_total_sent
X_tx_enriched[:, 15] = total_recv                                  # sender_total_received
X_tx_enriched[:, 16] = balance                                     # sender_balance
X_tx_enriched[:, 17] = avg_sent                                    # sender_avg_sent
X_tx_enriched[:, 18] = unique_sent                                 # sender_unique_receivers
X_tx_enriched[:, 19] = in_out_ratio                                # sender_in_out_ratio
X_tx_enriched[:, 20] = time_diff                                   # sender_active_duration_mins
X_tx_enriched = np.nan_to_num(X_tx_enriched, nan=0.0, posinf=0.0, neginf=0.0)
tx_enriched_populated = sum(1 for i in range(21) if (X_tx_enriched[:, i] != 0).any())
print(f'  TX-Level: {tx_enriched_populated}/21 features populated (was 9/21)')

# ── Student v2 (Gen 2): 71 features — 5/5 tabular now with REAL scores ──
STUDENT_FEATURES = student_features  # from Cell 6b

X_student_enriched = np.zeros((N, len(STUDENT_FEATURES)), dtype=np.float32)

student_tab_map = {
    'sender_total_transactions': sent_tnx + recv_tnx,
    'sender_total_sent': total_sent,
    'sender_total_received': total_recv,
    'sender_sophisticated_score': sophisticated_score,   # FATF pattern sum
    'sender_hybrid_score': hybrid_score,                 # REAL: 0.4*TeacherXGB + 0.3*pattern_boost + 0.3*soph
}

for i, feat_name in enumerate(STUDENT_FEATURES):
    if feat_name in student_tab_map:
        X_student_enriched[:, i] = student_tab_map[feat_name]

X_student_enriched = np.nan_to_num(X_student_enriched, nan=0.0, posinf=0.0, neginf=0.0)
student_enriched_populated = sum(1 for i in range(len(STUDENT_FEATURES)) if (X_student_enriched[:, i] != 0).any())
print(f'  Student v2: {student_enriched_populated}/{len(STUDENT_FEATURES)} features populated (was 3/{len(STUDENT_FEATURES)})')
print(f'    sender_sophisticated_score: sum of 7 FATF pattern scores')
print(f'    sender_hybrid_score:        0.4*Teacher_XGB + 0.3*pattern_boost + 0.3*soph (REAL, not proxy)')

# ── Store enriched data for the full comparison cell ──
enriched_data = {
    'X_tx_enriched': X_tx_enriched,
    'X_student_enriched': X_student_enriched,
    'X_teacher': X_teacher,
    'teacher_xgb_model': teacher_xgb_model,
    'teacher_xgb_raw': teacher_xgb_raw,
    'teacher_feature_names': teacher_feature_names,
    'teacher_mapped_count': teacher_mapped_count,
    'teacher_schema': teacher_schema,
    'tx_enriched_count': tx_enriched_populated,
    'student_enriched_count': student_enriched_populated,
    'sophisticated_score': sophisticated_score,
    'hybrid_score': hybrid_score,
    'pattern_count': pattern_count,
    'smurf_score': smurf_score,
    'fan_out_score': fan_out_score,
    'fan_in_score': fan_in_score,
    'layering_score': layering_score,
    'structuring_score': structuring_score,
    'velocity_score': velocity_score,
    'peeling_score': peeling_score,
    'degree': degree,
    'degree_centrality': degree_centrality,
    'clustering_approx': clustering_approx,
    'pattern_boost': pattern_boost,
    'soph_normalized': soph_normalized,
    'xgb_normalized': xgb_normalized,
}

# ── Summary ──
print(f'\n{"="*90}')
print(f'  FEATURE RECONSTRUCTION SUMMARY')
print(f'{"="*90}')
print(f'  {"Feature Source":<40} {"Before":>8} {"After":>8} {"Gain":>8}')
print(f'  {"─"*70}')
print(f'  {"TX-Level (21 features)":<40} {"9/21":>8} {f"{tx_enriched_populated}/21":>8} {f"+{tx_enriched_populated-9}":>8}')
print(f'  {"Student v2 tabular (5 of 71)":<40} {"3/5":>8} {f"5/5":>8} {"+2":>8}')
print(f'  {"hybrid_score source":<40} {"proxy":>8} {"Teacher":>8} {"REAL":>8}')
print(f'  {"FATF pattern scores":<40} {"0":>8} {"7":>8} {"+7":>8}')
print(f'  {"Graph properties (approx)":<40} {"0":>8} {"4":>8} {"+4":>8}')
print(f'  {"─"*70}')
print(f'  ✅ Feature reconstruction complete — hybrid_score uses REAL Teacher XGB')

  FEATURE RECONSTRUCTION — Replicating Training Pipeline on XBlock

  XBlock records: 9,841

── Loading Teacher XGB (Gen 1) for hybrid_score computation ──
  Loaded Teacher XGB (171 features)
  Features populated: 17/56 core features
  Teacher XGB raw probs: mean=0.000365, std=0.000054, min=0.000256, max=0.000401
  XGB normalized (0-100): mean=91.16, std=13.37

── Computing 7 FATF Pattern Scores ──
  SMURFING:      500 addresses flagged
  FAN_OUT:      1210 addresses flagged
  FAN_IN:       1331 addresses flagged
  LAYERING:     5120 addresses flagged
  STRUCTURING:   514 addresses flagged
  VELOCITY:      112 addresses flagged
  PEELING:      4834 addresses flagged

── Computing sender_sophisticated_score ──
  Addresses with any pattern: 6,188 (62.9%)
  Multi-pattern (≥2):         4,335 (44.1%)
  Score range: [0.0, 576.0], mean=187.0

── Computing sender_hybrid_score (with REAL Teacher XGB scores) ──
  XGB normalized (REAL Teacher):  mean=91.16, range=[63.80, 100.00]
  Pattern boost: 

In [13]:
# ============================================================================
# CELL 8: Full Risk Engine Stack — ALL Models + Rules + Combined vs XBlock
# ============================================================================
# Using ENRICHED features from the reconstruction cell above.
#   ML Generation 1 (Teacher):  XGB (171 features, ~31 mapped from XBlock)
#   ML Generation 2 (Student):  XGB, LGB, Ensemble (71 features, 5/5 tabular now mapped)
#   ML Generation 3 (TX-Level): XGB, LGB, Ensemble (21 features, 14 mapped)
#   Rule-based detection:       8 AML/pattern rules from integration_service.py
#   FATF Pattern Scores:        7 reconstructed scores from sophisticated_fraud_detection_ultra.py
#   Combined ML + Rules:        production-style 70/30 blend & OR-union

import xgboost as xgb
import lightgbm as lgb
import numpy as np
import pandas as pd
import json
import warnings
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from scipy.special import expit

warnings.filterwarnings('ignore')

print('='*90)
print('  FULL RISK ENGINE STACK — XBlock Phishing (N=9,841, fraud=22.14%)')
print('  With RECONSTRUCTED features (FATF patterns, sophisticated_score, hybrid_score)')
print('='*90)

# ── Re-use variables from previous cells ──
# df_xb, y_xb from Cell 6
# student_xgb, student_lgb, STUDENT_META_COEF, STUDENT_META_INTERCEPT from Cell 6b
# X_tx_enriched, X_student_enriched, enriched_data from Feature Reconstruction cell

# ============================================================================
# PART 1: Re-use Teacher XGB from Feature Reconstruction cell
# ============================================================================
print('\n── Teacher XGB (Gen 1, 171 features) — already loaded ──')
teacher_xgb = enriched_data['teacher_xgb_model']
teacher_feature_names = enriched_data['teacher_feature_names']
teacher_mapped_count = enriched_data['teacher_mapped_count']
teacher_schema = enriched_data['teacher_schema']
X_teacher = enriched_data['X_teacher']
teacher_xgb_prob = enriched_data['teacher_xgb_raw']  # REAL predictions, same as used for hybrid_score
print(f'  Features populated: {teacher_mapped_count}/{len(teacher_schema["core_features"])} core features')
print(f'  Teacher XGB probs: mean={teacher_xgb_prob.mean():.6f}, std={teacher_xgb_prob.std():.6f}')

# ============================================================================
# PART 2: TX-Level models with ENRICHED features (14/21 instead of 9/21)
# ============================================================================
print('\n── TX-Level Models with ENRICHED Features (Gen 3) ──')
print(f'  Features populated: {enriched_data["tx_enriched_count"]}/21 (was 9/21)')

# Use enriched feature matrix from reconstruction cell
tx_xgb_prob_enriched = xgb_model.predict_proba(X_tx_enriched)[:, 1]
tx_lgb_prob_enriched = lgb_model.predict(X_tx_enriched)
tx_lgb_prob_enriched = np.clip(tx_lgb_prob_enriched, 0, 1)

# TX-Level meta-ensemble with enriched features
tx_meta_input = np.column_stack([tx_xgb_prob_enriched, tx_lgb_prob_enriched])
tx_ensemble_prob_enriched = meta_model.predict_proba(tx_meta_input)[:, 1]

print(f'  TX-Level XGB (enriched) probs: mean={tx_xgb_prob_enriched.mean():.4f}, std={tx_xgb_prob_enriched.std():.4f}')
print(f'  TX-Level LGB (enriched) probs: mean={tx_lgb_prob_enriched.mean():.4f}, std={tx_lgb_prob_enriched.std():.4f}')
print(f'  TX-Level Ens (enriched) probs: mean={tx_ensemble_prob_enriched.mean():.4f}, std={tx_ensemble_prob_enriched.std():.4f}')

# Also keep original (9/21) for comparison
tx_xgb_prob_orig = xgb_model.predict_proba(X_xb)[:, 1]
tx_lgb_prob_orig = lgb_model.predict(X_xb)
tx_lgb_prob_orig = np.clip(tx_lgb_prob_orig, 0, 1)

# ============================================================================
# PART 3: Student v2 with ENRICHED features (5/5 tabular instead of 3/5)
# ============================================================================
print('\n── Student v2 Models with ENRICHED Features (Gen 2) ──')
print(f'  Features populated: {enriched_data["student_enriched_count"]}/{len(student_features)} (was 3/{len(student_features)})')
print(f'  NOW includes: sender_sophisticated_score, sender_hybrid_score')

# Use enriched feature matrix from reconstruction cell
try:
    student_xgb_prob_enriched = student_xgb.predict_proba(X_student_enriched)[:, 1]
except Exception:
    dmat = xgb.DMatrix(X_student_enriched, feature_names=[f'f{i}' for i in range(X_student_enriched.shape[1])])
    student_xgb_prob_enriched = student_xgb.predict(dmat)

student_lgb_prob_enriched = student_lgb.predict(X_student_enriched)
student_lgb_prob_enriched = np.clip(student_lgb_prob_enriched, 0, 1)

# Student v2 meta-ensemble (7 inputs: xgb, lgb, gatv2, graphsage, recon_err, vae, heuristic)
student_meta_enriched = np.column_stack([
    student_xgb_prob_enriched,
    student_lgb_prob_enriched,
    np.zeros(len(df_xb)),  # gatv2_score
    np.zeros(len(df_xb)),  # graphsage_score
    np.zeros(len(df_xb)),  # recon_err
    np.zeros(len(df_xb)),  # vae_score
    np.zeros(len(df_xb)),  # heuristic
])
student_logit_enriched = student_meta_enriched @ STUDENT_META_COEF + STUDENT_META_INTERCEPT
student_ensemble_prob_enriched = expit(student_logit_enriched)

print(f'  Student XGB (enriched) probs: mean={student_xgb_prob_enriched.mean():.4f}, std={student_xgb_prob_enriched.std():.4f}')
print(f'  Student LGB (enriched) probs: mean={student_lgb_prob_enriched.mean():.4f}, std={student_lgb_prob_enriched.std():.4f}')
print(f'  Student Ens (enriched) probs: mean={student_ensemble_prob_enriched.mean():.4f}, std={student_ensemble_prob_enriched.std():.4f}')

# ============================================================================
# PART 4: Rule-Based / FATF Pattern Detection
# ============================================================================
print('\n── Rule-Based & FATF Pattern Detection ──')

rule_flags = pd.DataFrame(index=df_xb.index)

# From integration_service.py / monitoring_rules.py (adapted for address aggregates)
sent_tnx_r     = pd.to_numeric(df_xb['Sent tnx'], errors='coerce').fillna(0)
unique_sent_r  = pd.to_numeric(df_xb['Unique Sent To Addresses'], errors='coerce').fillna(0)
unique_recv_r  = pd.to_numeric(df_xb['Unique Received From Addresses'], errors='coerce').fillna(0)
avg_val_sent_r = pd.to_numeric(df_xb['avg val sent'], errors='coerce').fillna(0)
total_sent_r   = pd.to_numeric(df_xb['total Ether sent'], errors='coerce').fillna(0)
total_recv_r   = pd.to_numeric(df_xb['total ether received'], errors='coerce').fillna(0)
balance_r      = pd.to_numeric(df_xb['total ether balance'], errors='coerce').fillna(0)
time_diff_r    = pd.to_numeric(df_xb['Time Diff between first and last (Mins)'], errors='coerce').fillna(0)
n_contracts_r  = pd.to_numeric(df_xb['Number of Created Contracts'], errors='coerce').fillna(0)

# 1. FAN_OUT
rule_flags['fan_out'] = (unique_sent_r >= 10).astype(int)
# 2. FAN_IN
rule_flags['fan_in'] = (unique_recv_r >= 10).astype(int)
# 3. SMURFING
rule_flags['smurfing'] = ((sent_tnx_r >= 10) & (avg_val_sent_r >= 0.1) & (avg_val_sent_r <= 1.0)).astype(int)
# 4. VELOCITY
days_active_r = time_diff_r / (24 * 60)
txs_per_day_r = sent_tnx_r / days_active_r.replace(0, 1)
rule_flags['velocity'] = (txs_per_day_r > 50).astype(int)
# 5. VALUE_CLUSTERING
rule_flags['value_clustering'] = ((avg_val_sent_r >= 9.0) & (avg_val_sent_r <= 10.0)).astype(int)
# 6. PASS_THROUGH
total_vol = total_sent_r + total_recv_r
pass_ratio_r = balance_r.abs() / total_vol.replace(0, 1)
rule_flags['pass_through'] = ((total_vol > 1.0) & (pass_ratio_r < 0.05)).astype(int)
# 7. HIGH_VALUE
rule_flags['high_value'] = (total_sent_r > 100).astype(int)
# 8. CONTRACT_CREATOR
rule_flags['contract_creator'] = (n_contracts_r > 0).astype(int)

print(f'\n  {"Rule":<20} {"Fired":>6} {"Rate":>8} {"Precision":>10} {"Recall":>8}')
print(f'  {"─"*60}')
total_rules_fired = np.zeros(len(df_xb))
for col in rule_flags.columns:
    fired = rule_flags[col].sum()
    rate = fired / len(df_xb)
    if fired > 0:
        prec = precision_score(y_xb, rule_flags[col], zero_division=0)
        rec = recall_score(y_xb, rule_flags[col], zero_division=0)
    else:
        prec = rec = 0.0
    total_rules_fired += rule_flags[col].values
    print(f'  {col:<20} {fired:>6} {rate:>8.2%} {prec:>10.4f} {rec:>8.4f}')

n_rules = len(rule_flags.columns)
rule_score = total_rules_fired / n_rules
rule_any = (total_rules_fired >= 1).astype(int)

# FATF sophisticated_score as a continuous predictor (from reconstruction cell)
soph_score_norm = enriched_data['soph_normalized'] / 100.0  # normalize to 0-1
soph_binary = (enriched_data['sophisticated_score'] > 0).astype(int)

print(f'\n  Any rule fired: {rule_any.sum():,} ({rule_any.mean():.1%})')
print(f'  FATF pattern detected: {soph_binary.sum():,} ({soph_binary.mean():.1%})')

# ============================================================================
# PART 5: Combined ML + Rules + FATF (production-style combinations)
# ============================================================================
print('\n── Combined ML + Rules + FATF Patterns ──')

# Best ML = TX-Level Ensemble with enriched features
ml_prob = tx_ensemble_prob_enriched

# A) Weighted blend: 70% ML + 30% Rules
combined_blend = 0.7 * ml_prob + 0.3 * rule_score

# B) ML + FATF sophisticated_score (continuous)
combined_ml_fatf = 0.6 * ml_prob + 0.4 * soph_score_norm

# C) ML + Rules + FATF triple blend
combined_triple = 0.5 * ml_prob + 0.25 * rule_score + 0.25 * soph_score_norm

# D) Rules boost: if FATF patterns or rules fire, boost ML probability
combined_boost = ml_prob.copy()
mask_fatf = enriched_data['pattern_count'] >= 2
mask_rule = total_rules_fired >= 2
mask_any = (enriched_data['pattern_count'] >= 1) | (total_rules_fired >= 1)
combined_boost[mask_fatf] = 0.3 * ml_prob[mask_fatf] + 0.7 * soph_score_norm[mask_fatf]
combined_boost[mask_rule & ~mask_fatf] = 0.4 * ml_prob[mask_rule & ~mask_fatf] + 0.6 * rule_score[mask_rule & ~mask_fatf]

# E) Student XGB (enriched) + FATF blend
combined_student_fatf = 0.6 * student_xgb_prob_enriched + 0.4 * soph_score_norm

# F) OR-union: flag if ANY layer fires
combined_or = np.maximum(np.maximum(ml_prob, rule_score), soph_score_norm)

# ============================================================================
# PART 6: Comprehensive Comparison Table
# ============================================================================
print('\n')
print('='*120)
print('  FULL RISK ENGINE COMPARISON — XBlock Phishing (N=9,841, fraud=22.14%)')
print('  With RECONSTRUCTED FATF patterns + enriched features')
print('='*120)

def eval_full(name, y_true, probs, feats_str='—'):
    roc = roc_auc_score(y_true, probs)
    ap = average_precision_score(y_true, probs)
    best_f1, best_t = 0, 0.5
    # Adaptive threshold search: use percentile-based candidates
    # This works regardless of probability scale (0.0004 or 0.8)
    pmin, pmax = probs.min(), probs.max()
    if pmax - pmin > 1e-10:  # Only search if there's variance
        # Combine: fixed grid + percentile-based grid spanning actual distribution
        fixed_thresholds = np.arange(0.01, 0.99, 0.01)
        percentile_thresholds = np.percentile(probs, np.arange(1, 100, 1))
        # Also add fine-grained thresholds around the distribution
        linear_thresholds = np.linspace(pmin, pmax, 200)
        all_thresholds = np.unique(np.concatenate([fixed_thresholds, percentile_thresholds, linear_thresholds]))
        for t in all_thresholds:
            f = f1_score(y_true, (probs >= t).astype(int), zero_division=0)
            if f > best_f1:
                best_f1, best_t = f, t
    preds_opt = (probs >= best_t).astype(int)
    prec = precision_score(y_true, preds_opt, zero_division=0)
    rec = recall_score(y_true, preds_opt, zero_division=0)
    cm = confusion_matrix(y_true, preds_opt)
    tp = cm[1, 1] if cm.shape[0] > 1 else 0
    fn = cm[1, 0] if cm.shape[0] > 1 else y_true.sum()
    fp = cm[0, 1] if cm.shape[1] > 1 else 0
    # Find which percentile the threshold corresponds to
    pctile = (probs < best_t).mean() * 100
    return {
        'name': name, 'roc': roc, 'ap': ap, 'f1': best_f1, 'thresh': best_t,
        'prec': prec, 'rec': rec, 'tp': tp, 'fn': fn, 'fp': fp, 'feats': feats_str,
        'pctile': pctile
    }

all_evals = []

# ─── Generation 1: Teacher ───
all_evals.append(eval_full('Teacher XGB (Gen1)',            y_xb, teacher_xgb_prob,         f'{teacher_mapped_count}/56'))

# ─── Generation 2: Student v2 ORIGINAL (3/5 tabular) ───
all_evals.append(eval_full('Student XGB orig (Gen2)',       y_xb, student_xgb_prob,         '3/71'))
all_evals.append(eval_full('Student LGB orig (Gen2)',       y_xb, student_lgb_prob,         '3/71'))
all_evals.append(eval_full('Student Ens orig (Gen2)',       y_xb, student_prob,             '3/71'))

# ─── Generation 2: Student v2 ENRICHED (5/5 tabular) ───
all_evals.append(eval_full('Student XGB +FATF (Gen2)',      y_xb, student_xgb_prob_enriched, '5/71'))
all_evals.append(eval_full('Student LGB +FATF (Gen2)',      y_xb, student_lgb_prob_enriched, '5/71'))
all_evals.append(eval_full('Student Ens +FATF (Gen2)',      y_xb, student_ensemble_prob_enriched, '5/71'))

# ─── Generation 3: TX-Level ORIGINAL (9/21) ───
all_evals.append(eval_full('TX-Level XGB orig (Gen3)',      y_xb, tx_xgb_prob_orig,         '9/21'))
all_evals.append(eval_full('TX-Level LGB orig (Gen3)',      y_xb, tx_lgb_prob_orig,         '9/21'))
all_evals.append(eval_full('TX-Level Ens orig (Gen3)',      y_xb, prob_xb,                  '9/21'))

# ─── Generation 3: TX-Level ENRICHED (14/21) ───
all_evals.append(eval_full('TX-Level XGB +feats (Gen3)',    y_xb, tx_xgb_prob_enriched,     '14/21'))
all_evals.append(eval_full('TX-Level LGB +feats (Gen3)',    y_xb, tx_lgb_prob_enriched,     '14/21'))
all_evals.append(eval_full('TX-Level Ens +feats (Gen3)',    y_xb, tx_ensemble_prob_enriched, '14/21'))

# ─── FATF / Rules only ───
all_evals.append(eval_full('Rules only (score)',            y_xb, rule_score,                '8 rules'))
all_evals.append(eval_full('FATF patterns (score)',         y_xb, soph_score_norm,           '7 FATF'))
all_evals.append(eval_full('FATF ≥1 (binary)',             y_xb, soph_binary.astype(float), '7 FATF'))

# ─── Combined ───
all_evals.append(eval_full('ML+Rules 70/30',               y_xb, combined_blend,            '14+8r'))
all_evals.append(eval_full('ML+FATF 60/40',                y_xb, combined_ml_fatf,           '14+7f'))
all_evals.append(eval_full('ML+Rules+FATF triple',         y_xb, combined_triple,            '14+8r+7f'))
all_evals.append(eval_full('ML+FATF boost',                y_xb, combined_boost,             '14+7f'))
all_evals.append(eval_full('Stu XGB+FATF 60/40',           y_xb, combined_student_fatf,      '5+7f'))
all_evals.append(eval_full('OR-union (max)',                y_xb, combined_or,                '14+8r+7f'))

# Print table
header = f'{"Model / Layer":<35} {"ROC-AUC":>8} {"PR-AUC":>8} {"Best F1":>8} {"θ*":>10} {"Pctile":>7} {"Prec":>7} {"Rec":>7} {"TP":>5} {"FN":>5} {"FP":>5} {"Features":>12}'
print(header)
print(f'{"─"*130}')

groups = [
    ('─── ML Gen 1 (Teacher, 171 feats) ───', [0]),
    ('─── ML Gen 2: Student v2 ORIGINAL (3/5 tabular) ───', [1, 2, 3]),
    ('─── ML Gen 2: Student v2 + FATF SCORES (5/5 tabular) ───', [4, 5, 6]),
    ('─── ML Gen 3: TX-Level ORIGINAL (9/21) ───', [7, 8, 9]),
    ('─── ML Gen 3: TX-Level + ENRICHED (14/21) ───', [10, 11, 12]),
    ('─── Rule / FATF Pattern Detection Only ───', [13, 14, 15]),
    ('─── Combined ML + Rules + FATF ───', [16, 17, 18, 19, 20, 21]),
]

for group_name, indices in groups:
    print(f'\n  {group_name}')
    for idx in indices:
        e = all_evals[idx]
        # Format threshold: scientific notation if very small
        t_str = f'{e["thresh"]:.2e}' if e['thresh'] < 0.001 else f'{e["thresh"]:.4f}'
        print(f'  {e["name"]:<33} {e["roc"]:>8.4f} {e["ap"]:>8.4f} {e["f1"]:>8.4f} {t_str:>10} {e["pctile"]:>6.1f}% {e["prec"]:>7.4f} {e["rec"]:>7.4f} {e["tp"]:>5} {e["fn"]:>5} {e["fp"]:>5} {e["feats"]:>12}')

print(f'\n  {"Random baseline":<33} {"0.5000":>8} {"0.2214":>8} {"—":>8} {"—":>10} {"—":>7} {"—":>7} {"—":>7} {"—":>5} {"—":>5} {"—":>5} {"—":>12}')
print(f'{"─"*130}')

# ============================================================================
# PART 7: Impact Analysis — Before vs After Feature Reconstruction
# ============================================================================
print('\n')
print('='*90)
print('  IMPACT ANALYSIS — Feature Reconstruction Uplift')
print('='*90)

comparisons = [
    ('Student XGB', all_evals[1], all_evals[4], '3/71 → 5/71'),
    ('Student LGB', all_evals[2], all_evals[5], '3/71 → 5/71'),
    ('Student Ens', all_evals[3], all_evals[6], '3/71 → 5/71'),
    ('TX-Level XGB', all_evals[7], all_evals[10], '9/21 → 14/21'),
    ('TX-Level LGB', all_evals[8], all_evals[11], '9/21 → 14/21'),
    ('TX-Level Ens', all_evals[9], all_evals[12], '9/21 → 14/21'),
]

print(f'\n  {"Model":<20} {"Original ROC":>12} {"Enriched ROC":>13} {"Δ ROC":>8} {"Orig F1":>8} {"Enrich F1":>10} {"Δ F1":>8}  Features')
print(f'  {"─"*100}')
for name, orig, enriched, feat_desc in comparisons:
    d_roc = enriched['roc'] - orig['roc']
    d_f1 = enriched['f1'] - orig['f1']
    arrow_roc = '▲' if d_roc > 0.001 else ('▼' if d_roc < -0.001 else '─')
    arrow_f1 = '▲' if d_f1 > 0.001 else ('▼' if d_f1 < -0.001 else '─')
    print(f'  {name:<20} {orig["roc"]:>12.4f} {enriched["roc"]:>13.4f} {arrow_roc}{abs(d_roc):>7.4f} {orig["f1"]:>8.4f} {enriched["f1"]:>10.4f} {arrow_f1}{abs(d_f1):>7.4f}  {feat_desc}')

# ============================================================================
# PART 8: Coverage Analysis — What does each layer uniquely catch?
# ============================================================================
print('\n')
print('='*90)
print('  COVERAGE ANALYSIS — Multi-Layer Detection')
print('='*90)

# Find best ML model (by ROC-AUC among enriched models)
best_ml_idx = max(range(len(all_evals[:13])), key=lambda i: all_evals[i]['roc'])
best_ml = all_evals[best_ml_idx]
best_ml_prob = [teacher_xgb_prob, student_xgb_prob, student_lgb_prob, student_prob,
                student_xgb_prob_enriched, student_lgb_prob_enriched, student_ensemble_prob_enriched,
                tx_xgb_prob_orig, tx_lgb_prob_orig, prob_xb,
                tx_xgb_prob_enriched, tx_lgb_prob_enriched, tx_ensemble_prob_enriched][best_ml_idx]

ml_preds = (best_ml_prob >= best_ml['thresh']).astype(int)
rule_preds = rule_any
fatf_preds = soph_binary

ml_tp_mask = (ml_preds == 1) & (y_xb == 1)
rule_tp_mask = (rule_preds == 1) & (y_xb == 1)
fatf_tp_mask = (fatf_preds == 1) & (y_xb == 1)

ml_tp = ml_tp_mask.sum()
rule_tp = rule_tp_mask.sum()
fatf_tp = fatf_tp_mask.sum()
total_fraud = y_xb.sum()

# Unique catches
ml_missed = (ml_preds == 0) & (y_xb == 1)
rules_catch_ml_miss = (ml_missed & (rule_preds == 1)).sum()
fatf_catch_ml_miss = (ml_missed & (fatf_preds == 1)).sum()
either_catch_ml_miss = (ml_missed & ((rule_preds == 1) | (fatf_preds == 1))).sum()

# Union: all layers
union_all = ((ml_preds == 1) | (rule_preds == 1) | (fatf_preds == 1)) & (y_xb == 1)

print(f'\n  Total fraud addresses: {total_fraud:,}')
print(f'\n  {"Detection Layer":<40} {"TP":>5} {"Recall":>8}')
print(f'  {"─"*55}')
best_ml_label = f"Best ML: {best_ml['name']} (θ={best_ml['thresh']:.2f})"
print(f'  {best_ml_label:<40} {ml_tp:>5} {ml_tp/total_fraud:>8.1%}')
print(f'  {"Rules (any ≥1)":<40} {rule_tp:>5} {rule_tp/total_fraud:>8.1%}')
print(f'  {"FATF patterns (any ≥1)":<40} {fatf_tp:>5} {fatf_tp/total_fraud:>8.1%}')
print(f'  {"─"*55}')
print(f'  {"ML + Rules + FATF (union)":<40} {union_all.sum():>5} {union_all.sum()/total_fraud:>8.1%}')
print(f'  {"─"*55}')

print(f'\n  Rules caught {rules_catch_ml_miss:,} fraud that ML missed')
print(f'  FATF caught {fatf_catch_ml_miss:,} fraud that ML missed')
print(f'  Either caught {either_catch_ml_miss:,} fraud that ML missed')
uplift = (union_all.sum() - ml_tp) / max(ml_tp, 1) * 100
print(f'  Coverage uplift from adding Rules+FATF to ML: +{uplift:.1f}%')

# Per-FATF-pattern contribution
print(f'\n  Per-FATF-Pattern Contribution (fraud caught by pattern, missed by ML):')
print(f'  {"Pattern":<20} {"Catches (ML miss)":>17} {"% of ML misses":>15}')
print(f'  {"─"*55}')
ml_fn_count = ml_missed.sum()
for pname, pscores in [('SMURFING', enriched_data['smurf_score']),
                        ('FAN_OUT', enriched_data['fan_out_score']),
                        ('FAN_IN', enriched_data['fan_in_score']),
                        ('LAYERING', enriched_data['layering_score']),
                        ('STRUCTURING', enriched_data['structuring_score']),
                        ('VELOCITY', enriched_data['velocity_score']),
                        ('PEELING', enriched_data['peeling_score'])]:
    catches = (ml_missed & (pscores > 0)).sum()
    pct = catches / max(ml_fn_count, 1) * 100
    if catches > 0:
        print(f'  {pname:<20} {catches:>17} {pct:>14.1f}%')

print(f'\n  ✅ Full risk engine stack evaluation complete')

  FULL RISK ENGINE STACK — XBlock Phishing (N=9,841, fraud=22.14%)
  With RECONSTRUCTED features (FATF patterns, sophisticated_score, hybrid_score)

── Teacher XGB (Gen 1, 171 features) — already loaded ──
  Features populated: 17/56 core features
  Teacher XGB probs: mean=0.000365, std=0.000054

── TX-Level Models with ENRICHED Features (Gen 3) ──
  Features populated: 12/21 (was 9/21)
  TX-Level XGB (enriched) probs: mean=0.0635, std=0.1139
  TX-Level LGB (enriched) probs: mean=0.1205, std=0.1512
  TX-Level Ens (enriched) probs: mean=0.0186, std=0.1017

── Student v2 Models with ENRICHED Features (Gen 2) ──
  Features populated: 5/71 (was 3/71)
  NOW includes: sender_sophisticated_score, sender_hybrid_score
  Student XGB (enriched) probs: mean=0.6490, std=0.4556
  Student LGB (enriched) probs: mean=0.7520, std=0.3558
  Student Ens (enriched) probs: mean=0.1275, std=0.0849

── Rule-Based & FATF Pattern Detection ──

  Rule                  Fired     Rate  Precision   Recall
  ────────

In [14]:
# Compact summary — key results with percentile thresholds
print('='*100)
print('  KEY RESULTS — Percentile-Based Thresholds (adaptive to each model\'s distribution)')
print('='*100)
print(f'\n  {"Model":<35} {"ROC":>7} {"F1":>7} {"θ*":>12} {"Pctile":>8} {"Prec":>7} {"Rec":>7} {"TP":>5}')
print(f'  {"─"*90}')

# Show top models from each generation
key_indices = [0, 1, 4, 7, 8, 10, 11]  # Teacher, Student XGB orig/enriched, TX-Level key models
for idx in key_indices:
    e = all_evals[idx]
    t_str = f'{e["thresh"]:.2e}' if e['thresh'] < 0.001 else f'{e["thresh"]:.4f}'
    print(f'  {e["name"]:<35} {e["roc"]:>7.4f} {e["f1"]:>7.4f} {t_str:>12} {e["pctile"]:>7.1f}% {e["prec"]:>7.4f} {e["rec"]:>7.4f} {e["tp"]:>5}')

print(f'\n  {"─"*90}')
print(f'  Teacher XGB prob range: [{teacher_xgb_prob.min():.6f}, {teacher_xgb_prob.max():.6f}]')
print(f'  → With adaptive threshold at percentile {all_evals[0]["pctile"]:.1f}%, Teacher now achieves F1={all_evals[0]["f1"]:.4f}')
print(f'\n  Key insight: ROC-AUC measures RANKING — the Teacher ranks well (0.75).')
print(f'  The old fixed threshold (0.01-0.99) missed this because all probs < 0.001.')
print(f'  Percentile thresholds unlock the ranking ability regardless of calibration.')

  KEY RESULTS — Percentile-Based Thresholds (adaptive to each model's distribution)

  Model                                   ROC      F1           θ*   Pctile    Prec     Rec    TP
  ──────────────────────────────────────────────────────────────────────────────────────────
  Teacher XGB (Gen1)                   0.7492  0.5288     3.87e-04    39.4%  0.3610  0.9881  2153
  Student XGB orig (Gen2)              0.6507  0.4669       0.0311    89.1%  0.7097  0.3479   758
  Student XGB +FATF (Gen2)             0.3740  0.3865       0.0049    11.9%  0.2418  0.9619  2096
  TX-Level XGB orig (Gen3)             0.5833  0.3838       0.0168    19.0%  0.2443  0.8940  1948
  TX-Level LGB orig (Gen3)             0.6385  0.4407       0.4700    91.9%  0.8221  0.3011   656
  TX-Level XGB +feats (Gen3)           0.6276  0.4037       0.0354    58.2%  0.3086  0.5833  1271
  TX-Level LGB +feats (Gen3)           0.6562  0.4387       0.0674    54.0%  0.3249  0.6751  1471

  ───────────────────────────────────